# 01 | IMTM: dos perfis agregados a descoberta e SHAP

**Papel nesta entrega:** Principal: campanha padronizada final

**Origem:** https://colab.research.google.com/drive/1JuFSgOmsr7bF_4lrZAPCr9z42NvDXLtS

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Os seletores de retomada/abertura automatica foram colocados em estado seguro nesta copia. Ative a etapa pretendida somente apos ler o protocolo; as quatro R4 ja foram consultadas no historico do trabalho.

As secoes 4-8 tratam entrada, qualidade, normalizacao, selecao e escores; 9-13, prioridade, bootstrap, caracterizacao e agrupamento; 14-17, regressao, SHAP, extremos e anotacoes; 18-19, R4 e fechamento.

Use o caderno 00 para posicionar apenas os arquivos documentais ausentes. Os CSVs agregados e resultados historicos nao sao substituidos. 

# IMTM_HepG2 | Cell Painting: dos perfis agregados aos candidatos e ao SHAP
### Um roteiro de análise para pesquisadores da área biomédica

**Autor do estudo:** Mario Costa Cruz. **Versão do roteiro:** 1.1-sites, adaptadores IMTM/FMP de 03/10/2026.

Este caderno pergunta: **quais tratamentos apresentam um perfil predominantemente mitocondrial que se repete?** Depois, pergunta se as medidas de uma réplica ajudam a prever a magnitude mitocondrial de outra e quais medidas sustentam essa previsão.

A análise começa nos **CSVs agregados por poço**, e não em uma tabela antiga de candidatos. Cada execução examina **todos os tratamentos presentes no mapa**. Nenhum código de composto, nome ou número esperado de candidatos foi escrito na regra. É possível encontrar zero, um, três ou muitos candidatos. A identificação farmacológica entra **depois**, sem orientar a seleção ou o aprendizado.

> **Generalizável não significa produzir nomes diferentes por obrigação.** Os mesmos dados, critérios e versões podem retornar os mesmos candidatos. Novos dados ou um novo protocolo previamente registrado podem produzir outra lista. Um identificador ausente da base de anotações continua na lista, com identidade não resolvida; não se inventa um nome.

O código é autocontido: não carrega scripts `.py` dos notebooks anteriores. As únicas entradas externas são dados e tabelas documentais. As células seguem a ordem da investigação; pequenas funções evitam repetir operações. Os comentários são em português e os identificadores do código não possuem acentos para facilitar sua edição.

**O que permanece no fluxo principal:** auditoria, mediana/MAD, seleção de medidas, RMS/P95, descoberta, caracterização, mapa multibloco, agrupamento exploratório, XGBoost e cinco comparadores, SHAP, auditoria de extremos, identidades e avaliação opcional de uma réplica reservada. As tentativas históricas de ZCA e calibração residual não são repetidas aqui e continuam nos cadernos anteriores.

**Limite do formato:** este roteiro aceita medidas CellProfiler dos objetos `Cells`, `Cyto` e `Nuc` e canais `AGP`, `DNA`, `ER`, `Mito`. Outro ensaio exige conferir os nomes e o mapa, não apenas trocar a pasta. Uma campanha contém uma linhagem/local: não misture conjuntos diferentes na pasta de entrada.

**Não é uma nova busca de moléculas na internet.** O caderno encontra candidatos entre os tratamentos efetivamente medidos. Uma identidade nova pode ser um código que não apareceu nas listas anteriores; a associação a nome e estrutura depende das fontes documentais fornecidas.

**Reanalise padronizada de IMTM_HepG2.** O núcleo de cálculos é o mesmo utilizado em MEDINA e USC; os modelos são treinados apenas neste local, a partir dos perfis agregados originais. Não são lidas listas históricas de candidatos nem reutilizados modelos antigos.

**R4 já foi consultada.** A separação R1–R3/R4 continua protegendo a sequência do processamento, mas esta execução é retrospectiva. Não recupera um teste intocado. Os resultados históricos permanecem em suas pastas; a nova execução usa `didatico_padronizado`.

O objetivo é reduzir diferenças de implementação antes da comparação entre quatro locais, não obter mais candidatos ou melhorar métricas por tentativas sucessivas. Os resultados podem permanecer iguais ou mudar; ambos serão documentados.


## 1. Preparar o ambiente - correcao de importacao

**Nesta versao, a instalacao e a analise ficam separadas por um reinicio obrigatorio quando algum pacote precisa ser instalado.** As versoes cientificas sao as mesmas da versao anterior; nenhum criterio de candidatos ou parametro dos modelos foi alterado.

Execute primeiro **apenas a celula de preparacao abaixo**. Quando ela informar `REINICIO NECESSARIO`, a parada e proposital: use **Ambiente de execucao > Reiniciar sessao** (em ingles, **Runtime > Restart session**) e execute novamente esta celula. Reiniciar limpa a memoria do Python; arquivos ja salvos no Drive permanecem. Nao use a opcao de excluir a maquina se precisar preservar arquivos temporarios locais.

Apos o reinicio, as versoes corretas sao reconhecidas sem reinstalacao. O teste verifica NumPy, SciPy, scikit-learn, XGBoost, SHAP, RDKit e as demais bibliotecas em um processo novo e na sessao atual. Prossiga somente depois de aparecer **AMBIENTE PRONTO**. Nesse momento, use **Executar tudo** ou continue pela secao 2. Uma segunda execucao na mesma sessao nao substitui o reinicio solicitado.

Se a importacao falhar mesmo no processo novo, a mensagem orienta a usar `FORCAR_REINSTALACAO = True` uma vez. Volte o valor para `False` antes de reiniciar para evitar reinstalacoes repetidas. Nao e necessario rebaixar NumPy para 1.x, substituir funcoes internas ou alterar os dados.

CSV e NPZ permanecem como saidas padrao; Parquet continua opcional. Os checkpoints preditivos exigem os mesmos dados, configuracao e versoes. Esta correcao foi limitada a preparacao do ambiente e a uma verificacao antes das importacoes cientificas; nao executa a campanha real.


In [ ]:
# Preparacao do ambiente: nenhuma leitura dos dados e nenhum ajuste do pipeline.
# Se houver instalacao, PARE e reinicie a sessao pelo menu antes de continuar.
import sys, subprocess, builtins
from importlib import import_module, invalidate_caches
from importlib.metadata import version, PackageNotFoundError

AMBIENTE_PRONTO = False
FORCAR_REINSTALACAO = False  # Somente para reparo, quando o teste em processo novo falhar.
PACOTES = [
    'numpy==2.3.5', 'pandas==2.2.3', 'scipy==1.17.0',
    'scikit-learn==1.8.0', 'xgboost==3.1.3', 'shap==0.50.0',
    'rdkit==2025.9.4', 'matplotlib==3.10.8', 'nbformat==5.10.4'
]
REINICIO = ('REINICIO NECESSARIO: use Ambiente de execucao > Reiniciar sessao '
            '(Runtime > Restart session). Depois execute novamente a secao 1. '
            'Nao basta executar de novo a celula sem reiniciar a sessao.')

# Testa inclusive numpy.char, que participa da cadeia de importacao do erro _slice.
TESTE_IMPORTACOES = '''
import numpy as np
import pandas, scipy, sklearn, xgboost, shap, matplotlib, nbformat
from scipy.stats import rankdata, spearmanr
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from rdkit import Chem
from threadpoolctl import threadpool_limits
assert np.char.add(['Cell'], ['Painting']).tolist() == ['CellPainting']
assert rankdata([3., 1., 2.]).tolist() == [3., 1., 2.]
assert sparse.csr_matrix(np.eye(2)).nnz == 2
assert Chem.MolFromSmiles('CCO') is not None
'''

def preparar_ambiente():
    if not (3, 11) <= sys.version_info[:2] <= (3, 14):
        raise RuntimeError('Use Python 3.11 a 3.14 com as versoes deste caderno.')
    if getattr(builtins, '_cellpainting_reinicio_pendente', False):
        raise RuntimeError(REINICIO)
    def instalada(nome):
        try:
            return version(nome)
        except PackageNotFoundError:
            return None
    esperadas = dict(p.split('==') for p in PACOTES)
    divergentes = [n for n, v in esperadas.items() if instalada(n) != v]
    if divergentes or FORCAR_REINSTALACAO:
        print('Preparando as versoes fixadas; nao prossiga para os dados nesta sessao.')
        comando = [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                   '--no-cache-dir', '--only-binary=:all:']
        if FORCAR_REINSTALACAO:
            comando.append('--force-reinstall')
        # Marca ANTES de instalar: ate uma instalacao interrompida exige reinicio.
        builtins._cellpainting_reinicio_pendente = True
        subprocess.check_call(comando + PACOTES)
        raise RuntimeError(REINICIO)
    # Primeiro verifica os arquivos instalados, em outro processo Python.
    teste = subprocess.run([sys.executable, '-c', TESTE_IMPORTACOES],
                           capture_output=True, text=True, timeout=180)
    if teste.returncode:
        print(teste.stdout)
        print(teste.stderr[-10000:])
        raise RuntimeError('Falha no teste das bibliotecas instaladas. Para reparar, '
                           'mude FORCAR_REINSTALACAO para True e execute esta celula. '
                           'Apos a instalacao, volte para False, reinicie a sessao '
                           'e execute novamente. Nao continue para os dados.')
    # Confere separadamente a memoria da sessao atual: pode conter versoes antigas.
    def normalizar(v):
        return '.'.join(str(int(x)) if x.isdigit() else x for x in str(v).split('.'))
    for pacote, esperada in esperadas.items():
        modulo = sys.modules.get('sklearn' if pacote == 'scikit-learn' else pacote)
        carregada = vars(modulo).get('__version__') if modulo is not None else None
        if carregada is not None and normalizar(carregada) != normalizar(esperada):
            builtins._cellpainting_reinicio_pendente = True
            raise RuntimeError(f'{pacote}: memoria={carregada}; disco={esperada}. ' + REINICIO)
    invalidate_caches()
    try:
        exec(TESTE_IMPORTACOES, {})
    except Exception as erro:
        builtins._cellpainting_reinicio_pendente = True
        raise RuntimeError('Os arquivos passam no teste, mas a sessao atual nao. ' + REINICIO) from erro
    return {nome: instalada(nome) for nome in esperadas}

VERSOES_AMBIENTE = preparar_ambiente()
AMBIENTE_PRONTO = True
print('AMBIENTE PRONTO. Agora execute a secao 2 e as seguintes, ou Executar tudo.')
print('Python:', sys.version.split()[0])
print(VERSOES_AMBIENTE)


## 2. A ficha do experimento: configuração local

Este caderno lê somente **IMTM_HepG2**, seu mapa documental e suas tabelas de identidades. No IMTM, `Metadata_Plate` contém `B1001_R1` e `Metadata_Batch` contém `HepG2_10uM`; a adaptação separa placa e réplica, como em MEDINA/USC.

R1–R3 definem as características, os P95, os candidatos e os modelos. `reserva_ja_consultada=True` registra o histórico real de R4. Deixe a abertura desativada até concluir o desenvolvimento; a seção 18 acompanha a lista calculada nesta execução, não uma lista anterior.

O parâmetro `PASTA_RETOMAR` só deve apontar para a pasta EXATA produzida por este novo caderno e mesma configuração. Não aponte para execuções históricas. Na retomada, refaça as células desde o início; os checkpoints compatíveis evitam retreinar modelos já salvos.

Os alertas de menos de 20 objetos e dominância maior que 80% não excluem poços. O filtro de contagem continua desativado; nenhum candidato é exigido por nome ou quantidade.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

RAIZ = Path('/content/drive/MyDrive/mba_tcc')
PASTA_DADOS = RAIZ / 'IMTM_HepG2'
PASTA_RESULTADOS = RAIZ / 'resultados_IMTM_HepG2' / 'didatico_padronizado'
FONTES_ZIP = RAIZ / 'pipelines_IMTM_HepG2' / 'fontes_documentais_IMTM_HepG2_padronizado.zip'
MAPA_CSV = None  # Para outro experimento: caminho de um CSV com seu proprio mapa.
MEMBRO_MAPA = 'annotations/2023-08-14_Annotation2_IMTM_HepG2.csv'
PADRAO_ARQUIVOS = '*_CP_Profiles_Aggregated.csv'
PASTA_RETOMAR = None

# Renomear somente os metadados, nunca as caracteristicas medidas.
COLUNAS = {
    'Metadata_Plate': 'placa', 'Metadata_Batch': 'replica',
    'Metadata_Well': 'poco', 'Metadata_EOS': 'composto',
    'Metadata_Concentration': 'dose', 'Metadata_Object_Count': 'n_objetos'
}
CFG = {
    'versao': 'didatico-1.1-sites', 'conjunto': 'IMTM_HepG2',
    'desenvolvimento': ['R1', 'R2', 'R3'], 'reservada': 'R4',
    'reserva_ja_consultada': True,
    'controles': ['DMSO'], 'bloco_alvo': 'Mito',
    'canais': ['AGP', 'DNA', 'ER', 'Mito'],
    'morfologia': ['Morfologia_Cells', 'Morfologia_Cyto', 'Morfologia_Nuc'],
    'min_controles': 10, 'fator_MAD': 1.4826, 'epsilon': 1e-18,
    'exportar_parquet': False,  # Opcional: NPZ + CSV ja preservam os dados completos.
    'correlacao_maxima': 0.90, 'quantil_referencia': 0.95,
    'min_replicas_ativas': 2, 'cosseno_minimo': 0.0,
    'alerta_objetos_abaixo': 20, 'alerta_dominancia_acima': 0.80,
    'min_objetos_inclusao': None,
    'seed': 20260929, 'n_bootstrap': 2000,
    'executar_agrupamento': True, 'k_candidatos': list(range(2, 9)),
    'n_subamostras': 20, 'executar_modelos': True,
    'folds_externos': 5, 'folds_internos': 4,
    'max_arvores': 800, 'paciencia': 40,
    'ridge_alphas': [1.0, 10.0, 100.0],
    'bootstrap_modelos': 1000,
    'min_fracao_finita_X': 0.99, 'min_IQR_X': 1e-12,
    'xgb': {'max_depth': 3, 'learning_rate': 0.04,
            'min_child_weight': 10, 'subsample': 0.8,
            'colsample_bytree': 0.8, 'reg_lambda': 10.0,
            'objective': 'reg:squarederror', 'tree_method': 'hist',
            'eval_metric': 'mae', 'n_jobs': 2}
}
# Opcional: identidade de outra biblioteca. Colunas: composto, name;
# opcionais: smiles, inchikey. O codigo precisa corresponder exatamente ao mapa.
IDENTIDADES_CSV = None
ALVOS_CSV = None  # Opcional: tabela longa com composto e registros de alvo/acao.
URL_PASTA_DADOS = 'https://drive.google.com/drive/folders/12fSaPsWXavic_hcmJFFy_hjY1hmQRvEj'
FONTES_SHA256 = '1d849078be7d3cd59d56f1f9400b48074ac46fe64c2c10aec30093dfd1c6ce2f'
CFG.update({'esquema_bruto': 'placa_inclui_replica', 'lote_bruto_esperado': 'HepG2_10uM',
            'validar_campanha_completa': True})

# Decisoes de proveniencia: sem consultar valores ou identidades dos candidatos.
CFG['arquivos_substituidos_documentados'] = {'2023-07-07_HepG2_10uM_B1001_R4_CP_Profiles_Aggregated.csv': '2023-07-17_HepG2_10uM_B1001_R4_CP_Profiles_Aggregated.csv'}
CFG['membro_registro_entrada'] = 'proveniencia/registro_entrada_IMTM.csv'
CFG['edicao_caderno'] = 'padronizacao_IMTM_FMP_20261003'
CFG['nucleo_calculos_sha256'] = '8abaf08d455b5fd14c30bf0f353b2cf364d7b63ad6e7e78ef0df33efdb5ca407'
CFG['nucleo_referencia_declarado_MEDINA_USC'] = '8dbfdd9da483686338e59eed7d02b4558822f9539f585345ae65886259788281'
CFG['nucleo_sha256'] = 'e0b0685600894c3d2edfc47de729c567819578d7129fdee1d2e5cfe54c30d562'


## 3. Ferramentas pequenas e registro da execução

As funções abaixo apenas padronizam a escrita das tabelas, o cálculo de integridade (SHA-256) e as figuras. A pasta de cada execução é nova. Uma retomada não permite mudar o protocolo ou as entradas. Dados antigos não são sobrescritos.

O código usa nomes curtos para os eixos experimentais: `placa`, `replica`, `poco`, `composto`, `dose`. Uma **condição** é a combinação de placa, composto e dose. O mesmo composto em doses diferentes constitui condições diferentes, mas todas ficam juntas na separação de treinamento e avaliação.

In [ ]:
# Protecao do ambiente; nao modifica criterios, dados ou modelos.
if not globals().get('AMBIENTE_PRONTO', False):
    raise RuntimeError('Execute a secao 1 e aguarde AMBIENTE PRONTO antes de continuar.')

import csv, io, json, hashlib, zipfile, shutil, warnings, platform, re
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from itertools import combinations, permutations
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score, adjusted_mutual_info_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import Ridge, LinearRegression
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor
import shap
from IPython.display import display, Markdown

CHAVE = ['placa', 'composto', 'dose_chave']
REPS = list(CFG['desenvolvimento'])
BLOCOS = CFG['canais'] + CFG['morfologia']
ALVO = CFG['bloco_alvo']
J_ALVO = BLOCOS.index(ALVO)
OUTROS = [j for j, b in enumerate(BLOCOS) if b != ALVO]
CONTROLES = {s.casefold() for s in CFG['controles']}
AVISOS = []
LEGENDAS = {}

def exigir(ok, mensagem):
    if not bool(ok):
        raise ValueError(mensagem)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for pedaco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(pedaco)
    return h.hexdigest()

def gravar_json(path, obj):
    def converter(x):
        if isinstance(x, np.ndarray): return x.tolist()
        if isinstance(x, np.generic): return x.item()
        if isinstance(x, Path): return str(x)
        raise TypeError(type(x).__name__)
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2,
                                    default=converter), encoding='utf-8')

def tabela(nome, dados):
    gravavel = dados.copy()
    if 'conjunto' not in gravavel: gravavel.insert(0, 'conjunto', CFG['conjunto'])
    gravavel.to_csv(SAIDA / 'tabelas' / (nome + '.csv'), index=False)
    return dados

def figura(nome, fig, legenda):
    fig.tight_layout()
    fig.savefig(SAIDA / 'figuras' / (nome + '.png'), dpi=180, bbox_inches='tight')
    fig.savefig(SAIDA / 'figuras' / (nome + '.svg'), bbox_inches='tight')
    LEGENDAS[nome] = legenda
    plt.show()
    plt.close(fig)

def controles(meta):
    return meta['composto'].str.casefold().isin(CONTROLES)

def dose_chave(valor):
    try:
        numero = Decimal(str(valor).strip())
    except InvalidOperation:
        raise ValueError('Dose nao numerica: ' + str(valor))
    exigir(numero.is_finite() and numero >= 0, 'Dose ausente, negativa ou nao finita.')
    return format(numero.normalize(), 'f')

def poco_padrao(valor):
    m = re.fullmatch(r'([A-Za-z]+)0*([1-9][0-9]*)', str(valor).strip())
    exigir(m is not None, 'Poco invalido: ' + str(valor))
    return m[1].upper() + str(int(m[2])).zfill(2)

def resumo_matriz(nome, meta, valores, nomes):
    np.savez_compressed(SAIDA / (nome + '.npz'), values=valores,
                        feature_names=np.asarray(nomes, dtype=str))
    tabela(nome + '_metadados', meta)
    if CFG['exportar_parquet']:
        import polars as pl  # Instale polars antes de ativar esta opcao.
        colunas_meta = {c: meta[c].to_numpy() for c in meta}
        quadro = pl.DataFrame(colunas_meta).hstack([
            pl.Series(n, valores[:, j], dtype=pl.Float64) for j, n in enumerate(nomes)])
        quadro.write_parquet(SAIDA / (nome + '.parquet'), compression='zstd')

exigir(len(REPS) >= 2 and len(set(REPS)) == len(REPS), 'Replicas de desenvolvimento invalidas.')
exigir(CFG['reservada'] not in REPS, 'A replica reservada nao pode participar do desenvolvimento.')
exigir(1 <= CFG['min_replicas_ativas'] <= len(REPS), 'Numero de replicas ativas invalido.')
exigir(ALVO in CFG['canais'], 'Escolha um canal-alvo existente; nao um identificador de composto.')
VERSOES = {'python': platform.python_version()}
for biblioteca in ['numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'shap', 'rdkit']:
    VERSOES[biblioteca] = version(biblioteca)
# Cada caderno le apenas sua pasta de perfis e sua propria pasta de resultados.
exigir(PASTA_DADOS.is_dir(), 'Pasta de perfis ausente: ' + str(PASTA_DADOS))
exigir(PASTA_DADOS.name == CFG['conjunto'], 'Pasta de entrada nao corresponde ao conjunto configurado.')
exigir(PASTA_DADOS.resolve() != PASTA_RESULTADOS.resolve(), 'Dados e resultados precisam ficar separados.')
if PASTA_RETOMAR:
    exigir(Path(PASTA_RETOMAR).resolve().is_relative_to(PASTA_RESULTADOS.resolve()),
           'Retomada fora da pasta de resultados deste local. Nao carregar outra campanha/local.')
if FONTES_ZIP:
    exigir(Path(FONTES_ZIP).is_file(), 'Pacote documental ausente: ' + str(FONTES_ZIP))
    if FONTES_SHA256:
        exigir(sha256(FONTES_ZIP) == FONTES_SHA256, 'Pacote documental alterado ou pertencente a outra edicao.')

SAIDA = Path(PASTA_RETOMAR) if PASTA_RETOMAR else (
    PASTA_RESULTADOS / (CFG['conjunto'] + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')))
SAIDA.mkdir(parents=True, exist_ok=bool(PASTA_RETOMAR))
for pasta in ['tabelas', 'figuras', 'modelos', 'shap', 'checkpoints']:
    (SAIDA / pasta).mkdir(exist_ok=True)
PROTOCOLO = {'config': json.loads(json.dumps(CFG)), 'colunas': COLUNAS.copy(), 'dados': str(PASTA_DADOS),
             'mapa_csv': str(MAPA_CSV), 'fontes_zip': str(FONTES_ZIP),
             'membro_mapa': MEMBRO_MAPA, 'padrao': PADRAO_ARQUIVOS,
             'identidades': str(IDENTIDADES_CSV), 'alvos': str(ALVOS_CSV), 'versoes': VERSOES, 'fontes_sha256_declarado': FONTES_SHA256, 'origem_perfis': URL_PASTA_DADOS}
if (SAIDA / 'protocolo.json').exists():
    exigir(json.loads((SAIDA / 'protocolo.json').read_text()) == PROTOCOLO,
           'A retomada tem configuracao ou versoes diferentes. Crie uma nova execucao.')
else:
    gravar_json(SAIDA / 'protocolo.json', PROTOCOLO)
HASH_PROTOCOLO = sha256(SAIDA / 'protocolo.json')
print('Resultados:', SAIDA)
print('Alvo:', ALVO, '| Desenvolvimento:', REPS, '| Reserva:', CFG['reservada'])

## 4. Conferir os arquivos e o mapa antes de olhar as medidas

A junção deve colocar o tratamento correto em cada poço. Duplicações, doses ausentes e poços sem correspondência interrompem a análise: escolher uma linha arbitrária poderia trocar a identidade de um resultado. Poços previstos mas ausentes são **registrados**, não inventados ou preenchidos.

Nesta passagem, dos arquivos da réplica reservada são lidos apenas cabeçalho e identificadores de placa, réplica e poço. Valores de medidas e contagens de objetos dessa réplica não entram na auditoria de desenvolvimento. Um arquivo que misture réplicas reservadas e de desenvolvimento deve ser separado antes de usar este caderno.

As colunas de medidas são inferidas dos arquivos de desenvolvimento, não de uma lista antiga. Para outro experimento, forneça um mapa próprio em `MAPA_CSV`; a biblioteca de compostos pode conter códigos novos, sem o prefixo EOS.

**Proveniência IMTM:** o README dos autores registra uma troca de identificação de B1001/R4 e B1007/R4 corrigida nas versões posteriores. A pasta ainda contém uma versão antiga de B1001/R4. Este caderno utiliza os mesmos 28 arquivos aprovados no registro de entrada histórico, verificando seus hashes. O arquivo antigo é registrado como substituído, não apagado. Nenhuma nova troca de placas é aplicada. Se aparecer outro arquivo ou hash, a execução para para revisão, sem escolher pela aparência dos resultados.


In [ ]:
def padronizar_metadados_brutos(quadro, path):
    """Traduzir apenas identificadores; nao transformar medidas ou inferir tratamentos."""
    quadro = quadro.copy()
    for col in ['placa', 'replica', 'poco']:
        exigir(col in quadro and quadro[col].notna().all(), 'Chave ausente: ' + str(path))
        quadro[col] = quadro[col].astype(str).str.strip()
    quadro['placa_na_fonte'] = quadro['placa']
    quadro['lote_na_fonte'] = quadro['replica']
    if CFG['esquema_bruto'] == 'placa_inclui_replica':
        partes = quadro['placa'].str.extract(r'^(B[0-9]+)_(R[0-9]+)$')
        exigir(partes.notna().all().all(), 'Esperado Metadata_Plate como B1001_R1: ' + path.name)
        exigir(quadro['lote_na_fonte'].eq(CFG['lote_bruto_esperado']).all(),
               'Lote/linhagem inesperado: ' + path.name)
        quadro['placa'], quadro['replica'] = partes[0], partes[1]
        arquivo = re.search(r'_(B[0-9]+)_(R[0-9]+)_CP_Profiles_Aggregated[.]csv$', path.name)
        exigir(arquivo is not None, 'Nome sem placa/replica: ' + path.name)
        placa_arquivo, replica_arquivo = arquivo[1], arquivo[2]
    elif CFG['esquema_bruto'] == 'replica_separada':
        exigir(quadro.placa.str.fullmatch(r'B[0-9]+').all(), 'Placa FMP invalida: ' + path.name)
        exigir(quadro.replica.str.fullmatch(r'R[0-9]+').all(), 'Replica FMP invalida: ' + path.name)
        arquivo = re.search(r'_(R[0-9]+)_(B[0-9]+)_CP_Profiles_Aggregated[.]csv$', path.name)
        exigir(arquivo is not None, 'Nome FMP sem replica/placa: ' + path.name)
        replica_arquivo, placa_arquivo = arquivo[1], arquivo[2]
    else:
        raise ValueError('Esquema de metadados nao reconhecido.')
    exigir(quadro.placa.eq(placa_arquivo).all() and quadro.replica.eq(replica_arquivo).all(),
           'Nome do arquivo e metadados discordam: ' + path.name)
    return quadro

def ler_documento_csv(path, membro=None):
    if membro:
        with zipfile.ZipFile(path) as z:
            blob = z.read(membro)
        return pd.read_csv(io.BytesIO(blob), dtype=str, keep_default_na=False)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

mapa = ler_documento_csv(MAPA_CSV if MAPA_CSV else FONTES_ZIP,
                         None if MAPA_CSV else MEMBRO_MAPA).rename(columns=COLUNAS)
NECESSARIAS = ['placa', 'replica', 'poco', 'composto', 'dose']
exigir(set(NECESSARIAS) <= set(mapa), 'Mapa sem colunas: ' + str(set(NECESSARIAS) - set(mapa)))
mapa = mapa[NECESSARIAS].copy()
for col in NECESSARIAS:
    mapa[col] = mapa[col].str.strip()
    exigir(mapa[col].ne('').all(), 'Mapa com valor vazio em ' + col)
mapa['poco'] = mapa['poco'].map(poco_padrao)
mapa['dose_chave'] = mapa['dose'].map(dose_chave)
CHAVE_POCO = ['placa', 'replica', 'poco']
exigir(not mapa.duplicated(CHAVE_POCO).any(), 'Mapa possui pocos duplicados.')

ARQUIVOS = sorted(PASTA_DADOS.glob(PADRAO_ARQUIVOS))
# Versoes obsoletas so sao omitidas por decisao documental EXPLICITA, nunca por escores.
ARQUIVOS_SUPERSEDIDOS = []
for antigo, atual in CFG.get('arquivos_substituidos_documentados', {}).items():
    anterior, substituto = PASTA_DADOS / antigo, PASTA_DADOS / atual
    exigir(substituto.is_file(), 'Falta o arquivo substituto documentado: ' + atual)
    if anterior.is_file():
        ARQUIVOS_SUPERSEDIDOS.append({'arquivo_nao_utilizado': antigo, 'arquivo_substituto': atual,
            'sha256_nao_utilizado': sha256(anterior),
            'motivo': 'Versao de arquivo substituida documentalmente; nao e filtro de pocos ou compostos.'})
        ARQUIVOS = [p for p in ARQUIVOS if p.name != antigo]
tabela('00_arquivos_substituidos_documentados', pd.DataFrame(ARQUIVOS_SUPERSEDIDOS,
    columns=['arquivo_nao_utilizado','arquivo_substituto','sha256_nao_utilizado','motivo']))
if CFG.get('membro_registro_entrada'):
    registro_fonte = ler_documento_csv(FONTES_ZIP, CFG['membro_registro_entrada'])
    exigir(registro_fonte.arquivo.is_unique, 'Manifesto de arquivos duplicado.')
    exigir(set(registro_fonte.arquivo) == {p.name for p in ARQUIVOS},
           'Os arquivos diferem da entrada historica aprovada. Rever proveniencia; nao escolher automaticamente.')
    verificados = []
    for row in registro_fonte.itertuples():
        atual = sha256(PASTA_DADOS / row.arquivo)
        verificados.append({'arquivo': row.arquivo, 'sha256_referencia': row.sha256,
                            'sha256_atual': atual, 'identico': atual == row.sha256})
    tabela('00_integridade_entrada_historica', pd.DataFrame(verificados))
    exigir(all(r['identico'] for r in verificados),
           'Algum perfil mudou desde o manifesto. Confira a tabela 00; nao prosseguir sem resolver.')
exigir(bool(ARQUIVOS), 'Nenhum perfil agregado encontrado na pasta informada.')
ORIGINAIS = {str(p): sha256(p) for p in ARQUIVOS}
for path in [MAPA_CSV, FONTES_ZIP, IDENTIDADES_CSV, ALVOS_CSV]:
    if path and Path(path).is_file(): ORIGINAIS[str(path)] = sha256(path)
if (SAIDA / 'entradas_sha256.json').exists():
    exigir(json.loads((SAIDA / 'entradas_sha256.json').read_text()) == ORIGINAIS,
           'Dados ou anotacoes mudaram. Nao misture a retomada com outra entrada.')
else:
    gravar_json(SAIDA / 'entradas_sha256.json', ORIGINAIS)

catalogo_arquivos, metadados_todos, NOMES, DEV_FILES, RES_FILES = [], [], None, [], []
colunas_chaves = [orig for orig, novo in COLUNAS.items() if novo in CHAVE_POCO]
for path in ARQUIVOS:
    with path.open(encoding='utf-8-sig', newline='') as f: cabecalho = next(csv.reader(f))
    exigir(len(cabecalho) == len(set(cabecalho)), 'Colunas duplicadas: ' + path.name)
    exigir(set(colunas_chaves) <= set(cabecalho), 'Chaves ausentes: ' + path.name)
    identificadores = pd.read_csv(path, usecols=colunas_chaves, dtype=str).rename(columns=COLUNAS)
    identificadores = padronizar_metadados_brutos(identificadores, path)
    for col in CHAVE_POCO:
        exigir(identificadores[col].notna().all(), 'Chave vazia em ' + path.name)
        identificadores[col] = identificadores[col].str.strip()
    identificadores['poco'] = identificadores['poco'].map(poco_padrao)
    exigir(not identificadores.duplicated(CHAVE_POCO).any(), 'Pocos duplicados: ' + path.name)
    replicas = set(identificadores.replica)
    e_dev = replicas <= set(REPS)
    e_res = replicas == {CFG['reservada']}
    exigir(e_dev or e_res, 'Arquivo mistura desenvolvimento/reserva ou replicas inesperadas: ' + path.name)
    medidas = sorted(n for n in cabecalho if re.match(r'^(Cells|Cyto|Nuc)_', n))
    if e_dev:
        exigir(bool(medidas), 'Nenhuma medida CellProfiler reconhecida: ' + path.name)
        if NOMES is None: NOMES = medidas
        exigir(medidas == NOMES, 'Esquemas de desenvolvimento diferentes: ' + path.name)
        DEV_FILES.append(path)
    else:
        RES_FILES.append(path)
    unidos = identificadores.merge(mapa, on=CHAVE_POCO, how='left', validate='one_to_one', indicator=True)
    exigir(unidos['_merge'].eq('both').all(), 'Poco sem tratamento no mapa: ' + path.name)
    unidos = unidos.drop(columns='_merge')
    unidos['arquivo'] = path.name
    metadados_todos.append(unidos)
    catalogo_arquivos.append({'arquivo': path.name, 'n_perfis': len(unidos),
        'replicas': ' | '.join(sorted(replicas)), 'medidas_no_cabecalho': len(medidas),
        'valores_liberados': e_dev, 'sha256': ORIGINAIS[str(path)]})
meta_inventario = pd.concat(metadados_todos, ignore_index=True)
exigir(not meta_inventario.duplicated(CHAVE_POCO).any(), 'O mesmo poco aparece em arquivos diferentes.')
exigir(set(REPS) <= set(meta_inventario.replica), 'Falta uma replica de desenvolvimento.')
previstos = mapa[mapa.replica.isin(REPS + [CFG['reservada']])]
faltantes = previstos.merge(meta_inventario[CHAVE_POCO], on=CHAVE_POCO, how='left', indicator=True)
faltantes = faltantes[faltantes['_merge'] == 'left_only'].drop(columns='_merge')
tabela('01_arquivos', pd.DataFrame(catalogo_arquivos))
tabela('01_pocos_ausentes', faltantes)
print('Arquivos:', len(ARQUIVOS), '| Medidas:', len(NOMES), '| Pocos ausentes:', len(faltantes))
# Verificacao de completude do desenho: pocos ausentes sao registrados, nao inventados.
if CFG['validar_campanha_completa']:
    esperado = set(map(tuple, previstos[['placa', 'replica']].drop_duplicates().to_numpy()))
    observado = set(map(tuple, meta_inventario[['placa', 'replica']].drop_duplicates().to_numpy()))
    exigir(esperado == observado, 'Placas/replicas ausentes ou extras: ' + str(esperado ^ observado))
    n_arquivos = meta_inventario.groupby(['placa', 'replica']).arquivo.nunique()
    exigir(n_arquivos.eq(1).all(), 'Mais de um arquivo por placa/replica: revisar antes de prosseguir.')
tabela('01_traducao_metadados', meta_inventario[['arquivo','lote_na_fonte','placa_na_fonte','placa','replica']].drop_duplicates())
tabela('01_cobertura_por_placa_replica', meta_inventario.groupby(['placa','replica']).agg(
    n_perfis=('poco','size'), n_controles=('composto',lambda s:s.str.casefold().isin(CONTROLES).sum())).reset_index())


## 5. Ler os perfis de desenvolvimento e auditar sua qualidade

Cada linha já resume os objetos de um poço. **Um perfil agregado de poucos objetos pode ser muito instável**, mas isso não permite afirmar automaticamente morte celular ou erro de segmentação. Por isso, salvamos a contagem e sua razão em relação aos DMSO da mesma placa/réplica.

A tabela de qualidade registra valores ausentes e infinitos por medida. Textos inesperados em colunas numéricas interrompem a execução. Dados ausentes continuam ausentes; os alvos e escores não serão imputados. O filtro opcional de contagem, se configurado, atua antes da normalização e suas exclusões ficam em tabela separada.

In [ ]:
def ler_valores(paths, nomes):
    metas, matrizes = [], []
    for path in paths:
        bruto = pd.read_csv(path, dtype={c: str for c in colunas_chaves}, low_memory=False)
        bruto = bruto.rename(columns=COLUNAS)
        bruto = padronizar_metadados_brutos(bruto, path)
        for c in CHAVE_POCO: bruto[c] = bruto[c].str.strip()
        bruto['poco'] = bruto['poco'].map(poco_padrao)
        indices = bruto[CHAVE_POCO].merge(mapa, on=CHAVE_POCO, how='left', validate='one_to_one')
        indices['arquivo'] = path.name
        indices['placa_na_fonte'] = bruto['placa_na_fonte'].to_numpy()
        indices['lote_na_fonte'] = bruto['lote_na_fonte'].to_numpy()
        indices['linha_csv'] = np.arange(2, len(bruto) + 2)
        indices['n_objetos'] = (pd.to_numeric(bruto['n_objetos'], errors='raise')
                                 if 'n_objetos' in bruto else np.nan)
        if 'n_objetos' in bruto:
            exigir(np.isfinite(indices.n_objetos).all() and (indices.n_objetos >= 0).all(),
                   'Contagem invalida em ' + path.name)
        matriz = np.full((len(bruto), len(nomes)), np.nan)
        for j, nome in enumerate(nomes):
            if nome in bruto: matriz[:, j] = pd.to_numeric(bruto[nome], errors='raise')
        metas.append(indices)
        matrizes.append(matriz)
    return pd.concat(metas, ignore_index=True), np.concatenate(matrizes)

meta, X_bruto = ler_valores(DEV_FILES, NOMES)
qualidade = pd.DataFrame({'medida': NOMES,
    'n_ausentes': np.isnan(X_bruto).sum(0), 'n_infinitos': np.isinf(X_bruto).sum(0)})
tabela('02_qualidade_numerica', qualidade)
exigir(not np.isinf(X_bruto).any(), 'Infinitos brutos encontrados: consulte a auditoria antes de continuar.')
ref_contagem = meta[controles(meta)].groupby(['placa', 'replica']).n_objetos.median()
meta['objetos_DMSO_mediana'] = [ref_contagem.get((p, r), np.nan) for p, r in zip(meta.placa, meta.replica)]
meta['objetos_relativos_DMSO'] = meta.n_objetos / meta.objetos_DMSO_mediana.replace(0, np.nan)
meta['alerta_poucos_objetos'] = meta.n_objetos.lt(CFG['alerta_objetos_abaixo'])
tabela('02_contagens_pocos', meta)
incluir = np.ones(len(meta), dtype=bool)
if CFG['min_objetos_inclusao'] is not None:
    exigir(meta.n_objetos.notna().all(), 'Filtro de contagem exige valores em todos os pocos.')
    incluir = meta.n_objetos.ge(CFG['min_objetos_inclusao']).to_numpy()
tabela('02_exclusoes_por_contagem', meta.loc[~incluir])
meta, X_bruto = meta.loc[incluir].reset_index(drop=True), X_bruto[incluir]
resumo_matriz('perfis_brutos_desenvolvimento', meta, X_bruto, NOMES)
print('Perfis incluidos:', len(meta), '| Controles:', int(controles(meta).sum()))
print('Alertas de poucos objetos (sem exclusao automatica):', int(meta.alerta_poucos_objetos.sum()))
display(meta.groupby(['placa', 'replica']).agg(n=('poco', 'size'), objetos_medianos=('n_objetos', 'median')))

## 6. Colocar as medidas na mesma referência: mediana e MAD dos DMSO

Imagine que uma medida de textura normalmente varia pouco entre controles e outra varia bastante. O mesmo desvio numérico não tem o mesmo significado nas duas. Subtraímos a **mediana** dos controles locais e dividimos por sua dispersão robusta: **1,4826 vezes o desvio absoluto mediano (MAD)**. Assim, o valor passa a representar um desvio em relação aos controles daquela placa e réplica.

MAD zero significa que essa escala não permite a divisão definida pelo protocolo. Não substituímos automaticamente por outro denominador nem preenchemos com zero. A escala fica inválida e é listada na auditoria. Uma característica com escala inválida no desenvolvimento não será selecionada. Se isso acontecer somente na reserva, a lista não será refeita para favorecer a avaliação.

**O que observar:** quantos controles existem, quais escalas falham e se a falha se concentra em determinadas placas ou famílias. Um MAD positivo, mas muito pequeno, também pode amplificar desvios; os parâmetros ficam disponíveis para inspeção.

In [ ]:
def normalizar_dmsos(meta, x, nomes):
    z = np.full(x.shape, np.nan)
    registros = []
    for (placa, replica), linhas in meta.groupby(['placa', 'replica']).indices.items():
        linhas = np.asarray(linhas)
        ctrl = linhas[controles(meta.iloc[linhas]).to_numpy()]
        exigir(len(ctrl) >= CFG['min_controles'], f'Controles insuficientes: {placa}/{replica}')
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            centro = np.nanmedian(x[ctrl], axis=0)
            escala = CFG['fator_MAD'] * np.nanmedian(np.abs(x[ctrl] - centro), axis=0)
        n = np.isfinite(x[ctrl]).sum(0)
        valido = (np.isfinite(centro) & np.isfinite(escala)
                  & (escala > CFG['epsilon']) & (n >= CFG['min_controles']))
        colunas = np.flatnonzero(valido)
        with np.errstate(over='ignore', divide='ignore', invalid='ignore'):
            z[np.ix_(linhas, colunas)] = ((x[np.ix_(linhas, colunas)] - centro[colunas])
                                          / (escala[colunas] + CFG['epsilon']))
        registros.append(pd.DataFrame({'placa': placa, 'replica': replica, 'medida': nomes,
            'mediana_DMSO': centro, 'MAD_escalado': escala,
            'n_controles_validos': n, 'escala_valida': valido}))
    return z, pd.concat(registros, ignore_index=True)

Z, parametros = normalizar_dmsos(meta, X_bruto, NOMES)
tabela('03_parametros_normalizacao', parametros)
tabela('03_escalas_invalidas', parametros.loc[~parametros.escala_valida])
resumo_matriz('perfis_MAD_desenvolvimento', meta, Z, NOMES)
display(parametros.groupby(['placa', 'replica']).escala_valida.agg(['sum', 'size']))

## 7. Selecionar características sem escolher compostos

O nome de uma medida informa **onde foi medida**, **qual propriedade descreve** e, quando presente, **qual canal foi utilizado**. Por exemplo, `Cells_Texture_..._Mito_...` descreve textura do canal Mito em objetos Cells. `Nuc` não significa nucléolo e não prova localização biológica exclusiva.

Primeiro retiramos medidas técnicas, constantes, não finitas ou sem escala válida nos controles. Depois reduzimos redundância dentro de cada bloco e família: quando duas medidas têm correlação de Spearman absoluta de pelo menos 0,90, mantemos um representante por ordem determinística. Não utilizamos nomes de compostos ou desempenho do XGBoost para essa escolha.

A lista resultante é própria desta execução. Não se exige encontrar 152 ou 178 medidas mitocondriais. Se um bloco obrigatório não tiver medidas, a rotina interrompe com uma explicação, em vez de criar um escore fictício.

**Canais e objetos:** `DNA` corresponde à marcação associada ao DNA; `ER`, ao retículo endoplasmático; `Mito`, à marcação mitocondrial; e `AGP`, à denominação de actina, Golgi e membrana plasmática do ensaio. Os blocos Cells, Cyto e Nuc são regiões/objetos de mensuração, não sete organelas independentes.

In [ ]:
def descrever_medida(nome):
    objeto, familia, metrica = nome.split('_', 2)
    canais = {'agp': 'AGP', 'dna': 'DNA', 'er': 'ER', 'mito': 'Mito', 'rna': 'RNA'}
    encontrados = sorted({canais[t] for t in metrica.lower().split('_') if t in canais})
    tecnica = (familia in {'Location', 'ImageQuality', 'Threshold'}
               or (familia == 'AreaShape' and metrica.startswith(
                   ('Center_', 'BoundingBoxMaximum_', 'BoundingBoxMinimum_')))
               or (familia == 'Neighbors' and 'ObjectNumber' in metrica))
    if tecnica: bloco, uso = 'TECNICO', 'excluir'
    elif len(encontrados) > 1: bloco, uso = 'Intercanais_' + '_'.join(encontrados), 'complementar'
    elif encontrados: bloco, uso = encontrados[0], 'principal'
    elif familia in {'AreaShape', 'Neighbors'}: bloco, uso = 'Morfologia_' + objeto, 'principal'
    else: bloco, uso = 'REVISAR', 'excluir'
    return {'medida': nome, 'objeto': objeto, 'familia': familia, 'bloco': bloco, 'uso': uso}

SEMANTICA = pd.DataFrame([descrever_medida(n) for n in NOMES])

def reduzir_redundancia(x, indices, semantica):
    mantidos, redundantes = [], {}
    for _, grupo in semantica.loc[indices].groupby(['bloco', 'familia'], sort=True):
        pos = grupo.index.to_numpy()
        ranks = rankdata(x[:, pos], axis=0)
        ranks -= ranks.mean(0)
        norms = np.linalg.norm(ranks, axis=0)
        exigir((norms > 0).all(), 'Caracteristica constante chegou ao filtro de correlacao.')
        ranks /= norms
        corr = ranks.T @ ranks
        escolhidos = []
        for j, indice in enumerate(pos):
            parecidos = [k for k in escolhidos if abs(corr[j, k]) >= CFG['correlacao_maxima']]
            if parecidos:
                redundantes[indice] = NOMES[pos[parecidos[0]]]
            else:
                escolhidos.append(j)
                mantidos.append(indice)
    return sorted(mantidos), redundantes

escala_ok = parametros.groupby('medida').escala_valida.all().reindex(NOMES).to_numpy()
finito = np.isfinite(Z).all(0)
variavel = np.zeros(len(NOMES), dtype=bool)
variavel[finito] = np.ptp(Z[:, finito], axis=0) > 0
selecionaveis = SEMANTICA.uso.ne('excluir').to_numpy() & escala_ok & finito & variavel
INDICES, redundantes = reduzir_redundancia(Z, np.flatnonzero(selecionaveis), SEMANTICA)
CATALOGO = SEMANTICA.copy()
CATALOGO['status'] = np.select([
    SEMANTICA.uso.eq('excluir'), ~escala_ok, ~finito, ~variavel],
    ['tecnica_ou_semantica_revisar', 'escala_DMSO_invalida', 'valor_nao_finito', 'constante'], default='selecionada')
CATALOGO.loc[list(redundantes), 'status'] = 'redundante'
CATALOGO['redundante_com'] = pd.Series(redundantes)
CATALOGO['selecionada'] = CATALOGO.index.isin(INDICES)
INDICES_BLOCO = {b: CATALOGO.index[CATALOGO.selecionada & CATALOGO.bloco.eq(b)].to_numpy() for b in BLOCOS}
for b, indices in INDICES_BLOCO.items():
    exigir(len(indices) > 0, 'Bloco sem medidas utilizaveis: ' + b)
tabela('04_catalogo_selecao', CATALOGO)
display(CATALOGO[CATALOGO.selecionada].groupby(['bloco', 'familia']).size().rename('n_medidas').to_frame())
print('Selecionadas:', len(INDICES), '| No alvo:', len(INDICES_BLOCO[ALVO]))

## 8. Resumir a magnitude: RMS e referência P95

A **raiz da média dos quadrados (RMS)** aumenta quando uma ou várias medidas se afastam dos controles. Valores positivos e negativos contribuem para a magnitude; seu sinal não é preservado no RMS. Calculamos então o **P95 dos RMS dos controles de desenvolvimento de cada placa** e dividimos: `A = RMS / P95`.

`A > 1` significa **acima dessa referência**, não probabilidade de efeito, valor-p, controle da taxa de falsas descobertas (FDR) ou disfunção mitocondrial. O mesmo A em dois locais não implica a mesma alteração biológica, pois suas referências e medidas podem diferir.

Esta etapa mostra ainda qual medida domina a soma quadrática de cada bloco por poço. Uma participação extrema, especialmente acompanhada de poucos objetos, exige inspeção. A tabela não remove casos. Nos controles, a proporção acima do limite usa somente observações avaliáveis; ausência de escore nunca é exibida como 0% de excedência.

In [ ]:
def rms_e_dominancia(x):
    n = len(x)
    resultado = np.full(n, np.nan)
    fracao = np.full(n, np.nan)
    dominante = np.full(n, -1, dtype=int)
    validos = np.isfinite(x).all(1)
    v = np.abs(x[validos])
    maior = v.max(1)
    proporcao = np.divide(v, maior[:, None], out=np.zeros_like(v), where=maior[:, None] > 0)
    quadrados = proporcao ** 2
    soma = quadrados.sum(1)
    resultado[validos] = maior * np.sqrt(quadrados.mean(1))
    fracao[validos] = np.divide(quadrados.max(1), soma, out=np.zeros(len(v)), where=soma > 0)
    dominante[validos] = np.where(soma > 0, quadrados.argmax(1), -1)
    return resultado, fracao, dominante

def escores(meta, z, referencias=None):
    saida = meta.copy()
    refs, diagnosticos = [], []
    for b, indices in INDICES_BLOCO.items():
        valor, fracao, dom = rms_e_dominancia(z[:, indices])
        saida['RMS_' + b] = valor
        if referencias is None:
            for placa, sub in saida[controles(saida)].groupby('placa'):
                for r in REPS:
                    exigir(np.isfinite(sub.loc[sub.replica == r, 'RMS_' + b]).sum() >= CFG['min_controles'],
                           f'DMSO avaliavel insuficiente: {placa}/{r}/{b}')
                limite = np.quantile(sub['RMS_' + b], CFG['quantil_referencia'], method='linear')
                exigir(np.isfinite(limite) and limite > 0, 'Referencia invalida: ' + placa + '/' + b)
                refs.append({'placa': placa, 'bloco': b, 'P95': limite,
                             'n_DMSO': len(sub), 'n_medidas': len(indices)})
        ref = pd.DataFrame(refs) if referencias is None else referencias
        lookup = ref.loc[ref.bloco == b].set_index('placa').P95
        saida['A_' + b] = valor / saida.placa.map(lookup)
        q = saida[['placa', 'replica', 'poco', 'composto', 'dose', 'n_objetos']].copy()
        q['bloco'] = b
        q['fracao_quadratica_maior_medida'] = fracao
        q['medida_dominante'] = [NOMES[indices[j]] if j >= 0 else '' for j in dom]
        q['alerta_dominancia'] = fracao > CFG['alerta_dominancia_acima']
        q['n_medidas_invalidas'] = (~np.isfinite(z[:, indices])).sum(1)
        diagnosticos.append(q)
    return saida, pd.DataFrame(refs) if referencias is None else referencias.copy(), pd.concat(diagnosticos, ignore_index=True)

def auditar_controles(scores):
    linhas = []
    for (p, r), grupo in scores[controles(scores)].groupby(['placa', 'replica']):
        for b in BLOCOS:
            a = grupo['A_' + b].to_numpy()
            valido = np.isfinite(a)
            linhas.append({'placa': p, 'replica': r, 'bloco': b, 'n_DMSO': len(a),
                'n_avaliaveis': int(valido.sum()), 'n_acima': int((a[valido] > 1).sum()),
                'taxa_pct': 100 * np.mean(a[valido] > 1) if valido.any() else np.nan,
                'mediana_A': np.median(a[valido]) if valido.any() else np.nan,
                'P95_A': np.quantile(a[valido], .95) if valido.any() else np.nan})
    return pd.DataFrame(linhas)

SCORES, REFERENCIAS, DOMINANCIA = escores(meta, Z)
AUD_CONTROLES = auditar_controles(SCORES)
tabela('05_escores_por_poco', SCORES)
tabela('05_referencias_congeladas', REFERENCIAS)
tabela('05_dominancia_por_poco', DOMINANCIA)
tabela('05_DMSO', AUD_CONTROLES)
fig, ax = plt.subplots(figsize=(9, 4))
for r in REPS:
    d = AUD_CONTROLES[(AUD_CONTROLES.replica == r) & (AUD_CONTROLES.bloco == ALVO)]
    ax.plot(d.placa, d.taxa_pct, marker='o', label=r)
ax.axhline(100 * (1 - CFG['quantil_referencia']), linestyle='--')
ax.set(xlabel='Placa', ylabel='% dos controles avaliaveis acima da referencia', title='Controles do desenvolvimento: ' + ALVO)
ax.legend()
figura('01_DMSO_desenvolvimento', fig, 'Os controles participaram da definicao das referencias. Esta figura nao e validacao independente.')

## 9. Descobrir candidatos: aplicar a mesma pergunta a todas as condições

Uma condição é priorizada quando tem todas as réplicas avaliáveis, o canal-alvo supera 1 em pelo menos o número de réplicas configurado, os outros blocos não superam 1 em nenhuma delas e todos os pares de perfis do alvo apresentam cosseno superior ao mínimo configurado. **Cosseno positivo significa direção concordante, não necessariamente alta semelhança.**

A tabela completa informa cada componente da regra e os motivos de não atendimento. Condições não avaliáveis são separadas daquelas que foram avaliadas e não passaram. Todos os candidatos encontrados são exportados: o limite usado em exibições e gráficos não reduz a lista.

A busca não consulta resultados passados, nomes de medicamentos, SHAP ou uma lista de códigos preferidos. Não presume que o número encontrado seja três. Após esta etapa, a lista da execução é congelada para a avaliação reservada: isso não limita a descoberta em campanhas novas; evita escolher candidatos depois de olhar a avaliação.

In [ ]:
def cosseno(a, b):
    if not np.isfinite(a).all() or not np.isfinite(b).all(): return np.nan
    sa, sb = np.max(np.abs(a)), np.max(np.abs(b))
    if sa == 0 or sb == 0: return np.nan
    a, b = a / sa, b / sb
    return float(np.clip(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)), -1, 1))

def agregar_matriz(meta, valores, nomes, condicoes, replicas):
    quadro = pd.concat([meta[CHAVE + ['replica']].reset_index(drop=True),
                        pd.DataFrame(valores, columns=nomes)], axis=1)
    # Perfil por condicao/replica: mediana dos pocos daquela condicao.
    agregado = quadro.groupby(CHAVE + ['replica'], sort=True)[nomes].median()
    indice = pd.MultiIndex.from_frame(condicoes[CHAVE])
    return np.stack([(agregado.xs(r, level='replica').reindex(indice).to_numpy(float)
                      if r in agregado.index.get_level_values('replica') else np.full((len(condicoes), len(nomes)), np.nan))
                     for r in replicas], axis=1)

condicoes = SCORES.loc[~controles(SCORES), CHAVE + ['dose']].drop_duplicates(CHAVE).sort_values(CHAVE).reset_index(drop=True)
condicoes.insert(0, 'id_condicao', np.arange(len(condicoes)))
COL_A = ['A_' + b for b in BLOCOS]
A_CUBO = agregar_matriz(meta, SCORES[COL_A].to_numpy(), COL_A, condicoes, REPS)
NOMES_ALVO = [NOMES[j] for j in INDICES_BLOCO[ALVO]]
Z_ALVO = agregar_matriz(meta, Z[:, INDICES_BLOCO[ALVO]], NOMES_ALVO, condicoes, REPS)
COS = np.asarray([[cosseno(v[a], v[b]) for a, b in combinations(range(len(REPS)), 2)] for v in Z_ALVO])
completas = np.isfinite(A_CUBO).all((1, 2)) & np.isfinite(Z_ALVO).all((1, 2)) & np.isfinite(COS).all(1)
passa_alvo = (A_CUBO[:, :, J_ALVO] > 1).sum(1) >= CFG['min_replicas_ativas']
passa_outros = (A_CUBO[:, :, OUTROS] <= 1).all((1, 2))
passa_cosseno = (COS > CFG['cosseno_minimo']).all(1)
condicoes['avaliavel'] = completas
condicoes['passa_alvo'] = passa_alvo
condicoes['passa_outros'] = passa_outros
condicoes['passa_cosseno'] = passa_cosseno
condicoes['cosseno_min'] = np.min(COS, axis=1)
condicoes['candidato'] = completas & passa_alvo & passa_outros & passa_cosseno
condicoes['status'] = np.where(~completas, 'nao_avaliavel', np.where(condicoes.candidato, 'candidato', 'nao_priorizado'))
condicoes['motivos'] = ['dados_ou_direcao_nao_avaliaveis' if not ok else ' | '.join(
    motivo for motivo, passou in [('magnitude_alvo', a), ('outros_blocos', o), ('direcao', c)] if not passou)
    for ok, a, o, c in zip(completas, passa_alvo, passa_outros, passa_cosseno)]
for j, bloco in enumerate(BLOCOS):
    for k, r in enumerate(REPS): condicoes['A_' + bloco + '_' + r] = A_CUBO[:, k, j]
    condicoes['A_' + bloco + '_mediana'] = np.median(A_CUBO[:, :, j], axis=1)
# Flags de qualidade acompanham o candidato; nao mudam silenciosamente a regra.
baixa = SCORES[~controles(SCORES)].groupby(CHAVE).alerta_poucos_objetos.any()
condicoes['alerta_poucos_objetos'] = [bool(baixa.get(tuple(row), False)) for row in condicoes[CHAVE].to_numpy()]
candidatos = condicoes[condicoes.candidato].copy()
tabela('06_todas_condicoes', condicoes)
tabela('06_candidatos_descobertos', candidatos)
tabela('06_condicoes_nao_avaliaveis', condicoes[~condicoes.avaliavel])
print('Condicoes analisadas:', len(condicoes), '| Candidatos encontrados:', len(candidatos))
display(candidatos[['id_condicao', 'placa', 'composto', 'dose', 'A_' + ALVO + '_mediana', 'cosseno_min', 'alerta_poucos_objetos']])

## 10. A decisão depende muito dos controles escolhidos?

Reamostramos os DMSO dentro de cada réplica, mantendo juntos os blocos do mesmo poço. O P95 é recalculado em cada repetição, e registramos a frequência com que cada candidato continua atendendo à regra. O perfil do tratamento e sua direção não são modificados.

Uma permanência de 90% significa que a regra resistiu a 90% dessas reamostragens. **Não significa 90% de probabilidade de confirmação biológica ou em R4.** Nenhum candidato é eliminado por baixa permanência. Se não houver candidatos, a tabela será vazia e o restante do pipeline continuará.

In [ ]:
def bootstrap_candidatos(candidatos, scores):
    resultado = []
    rng = np.random.default_rng(CFG['seed'])
    campos = ['RMS_' + b for b in BLOCOS]
    for placa, grupo in candidatos.groupby('placa', sort=True):
        pools = [scores[(scores.placa == placa) & (scores.replica == r) & controles(scores)][campos].to_numpy() for r in REPS]
        magnitudes = agregar_matriz(meta, scores[campos].to_numpy(), campos, grupo, REPS)
        vezes = np.zeros(len(grupo), dtype=int)
        for _ in range(CFG['n_bootstrap']):
            amostra = np.concatenate([a[rng.integers(0, len(a), len(a))] for a in pools])
            limites = np.quantile(amostra, CFG['quantil_referencia'], axis=0, method='linear')
            a = magnitudes / limites
            vezes += ((a[:, :, J_ALVO] > 1).sum(1) >= CFG['min_replicas_ativas']) & (a[:, :, OUTROS] <= 1).all((1, 2))
        for i, row in enumerate(grupo.itertuples()):
            resultado.append({'id_condicao': row.id_condicao, 'permanencia_pct': 100 * vezes[i] / CFG['n_bootstrap']})
    return pd.DataFrame(resultado, columns=['id_condicao', 'permanencia_pct'])

permanencia = bootstrap_candidatos(candidatos, SCORES)
candidatos = candidatos.merge(permanencia, on='id_condicao', how='left', validate='one_to_one')
tabela('07_candidatos_e_bootstrap', candidatos)
CONGELAMENTO = {'protocolo_sha256': HASH_PROTOCOLO, 'alvo': ALVO,
    'medidas_por_bloco': {b: [NOMES[j] for j in indices] for b, indices in INDICES_BLOCO.items()},
    'referencias': REFERENCIAS.to_dict('records'), 'candidatos': candidatos[CHAVE + ['id_condicao']].to_dict('records')}
if (SAIDA / 'descoberta_congelada.json').exists():
    exigir(json.loads((SAIDA / 'descoberta_congelada.json').read_text()) == CONGELAMENTO,
           'A descoberta mudou numa retomada. Crie outra execucao.')
else:
    gravar_json(SAIDA / 'descoberta_congelada.json', CONGELAMENTO)
HASH_DESCOBERTA = sha256(SAIDA / 'descoberta_congelada.json')
print('Lista congelada desta execucao:', len(candidatos), 'candidatos. Nenhum foi escolhido por nome.')

## 11. Caracterizar os candidatos encontrados, não uma lista antiga

Agora descrevemos **quais propriedades participam da magnitude** de cada candidato. Para cada medida, calculamos sua fração na soma das médias dos desvios normalizados ao quadrado das réplicas de desenvolvimento. Apresentamos os valores com sinal, as 20 maiores participações e os resumos por família.

Essa contribuição não é SHAP: ela descreve o perfil medido, enquanto SHAP explicará uma previsão do modelo. Famílias com mais medidas podem acumular mais contribuição. Por isso, salvamos também o número de medidas e o RMS por medida da família. Todas as medidas continuam no cálculo; top20 é apenas exibição.

In [ ]:
def caracterizar(condicoes_selecionadas, cubo_z, replicas, prefixo):
    linhas, familias = [], []
    sem = SEMANTICA.iloc[INDICES_BLOCO[ALVO]].reset_index(drop=True)
    for row in condicoes_selecionadas.itertuples():
        valores = cubo_z[row.id_condicao]
        if not np.isfinite(valores).all(): continue
        escala = np.abs(valores).max()
        q = np.mean((valores / escala) ** 2, axis=0) if escala > 0 else np.zeros(len(sem))
        contribuicao = 100 * q / q.sum() if q.sum() > 0 else np.zeros(len(q))
        ordem = np.argsort(-contribuicao, kind='stable')
        rank = np.empty(len(q), int)
        rank[ordem] = np.arange(1, len(q) + 1)
        for j, medida in enumerate(NOMES_ALVO):
            rec = {'id_condicao': row.id_condicao, 'composto': row.composto, 'placa': row.placa,
                   'medida': medida, 'familia': sem.iloc[j].familia,
                   'contribuicao_quadratica_pct': contribuicao[j], 'ordem_exibicao': rank[j]}
            rec.update({r: valores[k, j] for k, r in enumerate(replicas)})
            linhas.append(rec)
        for familia, f in sem.groupby('familia'):
            ids = f.index.to_numpy()
            familias.append({'id_condicao': row.id_condicao, 'composto': row.composto, 'familia': familia,
                'n_medidas': len(ids), 'contribuicao_pct': contribuicao[ids].sum(),
                'RMS_por_medida': float(escala * np.sqrt(q[ids].mean()))})
    completo = pd.DataFrame(linhas, columns=['id_condicao', 'composto', 'placa', 'medida', 'familia',
                                            'contribuicao_quadratica_pct', 'ordem_exibicao'] + list(replicas))
    tabela(prefixo + '_medidas', completo)
    if prefixo == '08_assinaturas':
        tabela(prefixo + '_top20', completo[completo.ordem_exibicao <= 20])
    por_familia = pd.DataFrame(familias, columns=['id_condicao', 'composto', 'familia', 'n_medidas', 'contribuicao_pct', 'RMS_por_medida'])
    tabela(prefixo + '_familias', por_familia)
    return completo, por_familia

CARACTERISTICAS_CANDIDATOS, FAMILIAS_CANDIDATOS = caracterizar(candidatos, Z_ALVO, REPS, '08_assinaturas')
if len(FAMILIAS_CANDIDATOS): display(FAMILIAS_CANDIDATOS)
else: print('Nenhum candidato para caracterizar. As demais analises continuam com a coorte completa.')

## 12. O perfil é restrito ao alvo ou acompanha outros blocos?

A análise multibloco volta a **todas as condições completas**, não apenas aos candidatos. Conta em quantas réplicas cada bloco excede a referência. Pares e trios exigem excedência **junta nas mesmas réplicas**: Mito alto em R1/R2 e DNA alto em R2/R3 não representam duas repetições de Mito+DNA.

Os padrões exatos descrevem toda a configuração dos blocos. As combinações inclusivas permitem outras excedências. Esses resultados contextualizam a regra, sem substituir candidatos nem demonstrar causalidade entre estruturas.

In [ ]:
validos = np.flatnonzero(np.isfinite(A_CUBO).all((1, 2)))
A_COMPLETO = A_CUBO[validos]
META_COMPLETO = condicoes.iloc[validos].reset_index(drop=True)
excedencias = []
for j, b in enumerate(BLOCOS):
    contagem = (A_COMPLETO[:, :, j] > 1).sum(1)
    excedencias.append({'bloco': b, 'n_condicoes': len(validos),
        'n_em_pelo_menos_2': int((contagem >= 2).sum()),
        'n_em_todas': int((contagem == len(REPS)).sum())})
EXCEDENCIAS = tabela('09_excedencia_blocos', pd.DataFrame(excedencias))
combinacoes = []
for tamanho in [2, 3]:
    for indices in combinations(range(len(BLOCOS)), tamanho):
        juntas = (A_COMPLETO[:, :, list(indices)] > 1).all(2)
        combinacoes.append({'blocos': ' + '.join(BLOCOS[j] for j in indices),
                            'n_simultanea_2rep': int((juntas.sum(1) >= 2).sum()),
                            'denominador': len(validos)})
tabela('09_combinacoes_simultaneas', pd.DataFrame(combinacoes))
recorrencias = []
for i, perfil in enumerate(A_COMPLETO):
    padroes = [tuple(np.flatnonzero(linha > 1)) for linha in perfil]
    repetidos = [p for p in dict.fromkeys(padroes) if padroes.count(p) >= 2]
    # Com mais replicas pode haver mais de um padrao recorrente; nenhum e omitido.
    descricao = ' ; '.join(' + '.join(BLOCOS[j] for j in p) or 'nenhum_bloco' for p in repetidos)
    recorrencias.append({'id_condicao': int(META_COMPLETO.iloc[i].id_condicao),
                         'padroes_recorrentes': descricao or 'sem_padrao_recorrente'})
tabela('09_padroes_recorrentes', pd.DataFrame(recorrencias))
display(EXCEDENCIAS)

## 13. Agrupamentos exploratórios: não confundir um grupo extremo com um fenótipo

Esta etapa organiza as condições por **magnitude global** ou por **contrastes entre os quatro canais**. O K-means testa os números de grupos configurados, usando silhouette como critério interno. Um silhouette elevado pode resultar da separação de pouquíssimas condições extremas; por isso, os tamanhos dos grupos, as contagens de objetos e os escores devem ser lidos juntos.

A estabilidade é examinada sob subamostragem de compostos e em rotações entre réplicas. **ARI (índice de Rand ajustado)** compara atribuições a grupos, não acurácia. **AMI (informação mútua ajustada) com placa** descreve associação, não porcentagem de variação técnica. Uma tabela adicional mostra a proporção que mantém o grupo nas réplicas individuais, com o modelo de referência fixo; esse valor pode ser dominado pelo maior grupo.

Esta célula inclui análise de componentes principais (PCA) apenas como visualização. Não são escolhidos grupos com base nos candidatos e nenhum grupo se transforma em rótulo biológico do XGBoost.

In [ ]:
def representar(a, espaco):
    x = np.log1p(a)
    if espaco == 'contrastes': x = x - x.mean(1, keepdims=True)
    return x

def ajustar_grupos(a, espaco, k):
    x = representar(a, espaco)
    mediana = np.median(x, axis=0)
    iqr = np.quantile(x, .75, axis=0) - np.quantile(x, .25, axis=0)
    iqr = np.where(iqr > 1e-12, iqr, 1.0)
    modelo = KMeans(n_clusters=k, n_init=30, max_iter=500, random_state=CFG['seed'])
    with threadpool_limits(limits=2): modelo.fit((x - mediana) / iqr)
    return modelo, mediana, iqr

def projetar_grupos(a, espaco, ajuste):
    modelo, centro, escala = ajuste
    return modelo.predict((representar(a, espaco) - centro) / escala)

AGRUPAMENTOS = []
if CFG['executar_agrupamento'] and len(validos) > max(CFG['k_candidatos']):
    principal = A_COMPLETO[:, :, :len(CFG['canais'])]
    mediana = np.median(principal, axis=1)
    for espaco in ['magnitude', 'contrastes']:
        busca = []
        for k in CFG['k_candidatos']:
            ajuste = ajustar_grupos(mediana, espaco, k)
            rotulos = ajuste[0].labels_
            x = (representar(mediana, espaco) - ajuste[1]) / ajuste[2]
            valor = (silhouette_score(x, rotulos, sample_size=min(5000, len(x)), random_state=CFG['seed'])
                     if 1 < len(np.unique(rotulos)) < len(x) else -1.0)
            busca.append({'k': k, 'silhouette': valor, 'n_menor': int(np.bincount(rotulos).min())})
        busca = pd.DataFrame(busca).sort_values(['silhouette', 'k'], ascending=[False, True])
        k = int(busca.iloc[0].k)
        ajuste = ajustar_grupos(mediana, espaco, k)
        rotulos = ajuste[0].labels_
        tabela('10_busca_k_' + espaco, busca)
        atribuicoes = META_COMPLETO[['id_condicao', 'placa', 'composto', 'dose']].copy()
        atribuicoes['grupo'] = rotulos
        tabela('10_grupos_' + espaco, atribuicoes)
        perfis = pd.DataFrame(mediana, columns=CFG['canais']).assign(grupo=rotulos)
        tabela('10_perfis_' + espaco, perfis.groupby('grupo').median().reset_index())
        rotacoes = []
        for destino, r in enumerate(REPS):
            treino = np.median(np.delete(principal, destino, axis=1), axis=1)
            outro_ajuste = ajustar_grupos(treino, espaco, k)
            previsto = projetar_grupos(principal[:, destino], espaco, outro_ajuste)
            rotacoes.append({'replica': r, 'ARI': adjusted_rand_score(outro_ajuste[0].labels_, previsto)})
        rng = np.random.default_rng(CFG['seed'])
        compostos = np.unique(META_COMPLETO.composto)
        estabilidade = []
        for b in range(CFG['n_subamostras']):
            n_sub = min(len(compostos), max(k + 1, int(.8 * len(compostos))))
            escolhidos = rng.choice(compostos, n_sub, replace=False)
            mask = META_COMPLETO.composto.isin(escolhidos).to_numpy()
            sub = ajustar_grupos(mediana[mask], espaco, k)
            estabilidade.append({'repeticao': b, 'ARI': adjusted_rand_score(rotulos[mask], sub[0].labels_)})
        projecoes = np.column_stack([projetar_grupos(principal[:, j], espaco, ajuste) for j in range(len(REPS))])
        tamanhos = pd.Series(rotulos).value_counts().sort_index()
        tabela('10_tamanho_grupos_' + espaco, tamanhos.rename_axis('grupo').rename('n').reset_index())
        tabela('10_rotacoes_' + espaco, pd.DataFrame(rotacoes))
        tabela('10_subamostras_' + espaco, pd.DataFrame(estabilidade))
        AGRUPAMENTOS.append({'espaco': espaco, 'k': k, 'silhouette': busca.iloc[0].silhouette,
            'menor_grupo': int(tamanhos.min()), 'ARI_replicas_mediano': np.median([z['ARI'] for z in rotacoes]),
            'AMI_placa': adjusted_mutual_info_score(META_COMPLETO.placa, rotulos),
            'mesmo_grupo_todas_replicas_pct': 100 * (projecoes == projecoes[:, [0]]).all(1).mean()})
        if tamanhos.min() / len(mediana) < .01:
            AVISOS.append('Agrupamento ' + espaco + ': grupo menor que 1% da coorte; verificar extremos.')
        x = (representar(mediana, espaco) - ajuste[1]) / ajuste[2]
        pca = PCA(n_components=2).fit(x)
        coordenadas = pca.transform(x)
        tabela('10_PCA_' + espaco, atribuicoes.assign(PC1=coordenadas[:, 0], PC2=coordenadas[:, 1]))
        tabela('10_PCA_cargas_' + espaco, pd.DataFrame(pca.components_.T, index=CFG['canais'], columns=['PC1', 'PC2']).rename_axis('canal').reset_index())
        fig, ax = plt.subplots(figsize=(7, 5))
        for g in sorted(set(rotulos)):
            m = rotulos == g
            ax.scatter(coordenadas[m, 0], coordenadas[m, 1], s=10, alpha=.5, label=f'Grupo {g}: n={m.sum()}')
        ax.set(xlabel='Componente principal 1', ylabel='Componente principal 2', title=espaco.capitalize())
        ax.legend()
        figura('02_PCA_' + espaco, fig, 'Projecao exploratoria. Os grupos nao sao classes biologicas; extremos e tamanhos devem ser examinados.')
else:
    print('Agrupamento desativado ou coorte insuficiente. A descoberta nao depende dele.')
RESUMO_GRUPOS = tabela('10_resumo_agrupamentos', pd.DataFrame(AGRUPAMENTOS))

## 14. A pergunta preditiva é diferente da descoberta

Usaremos as medidas de uma réplica para prever `log(1+A_alvo)` de outra réplica da mesma condição. A transformação comprime a amplitude do alvo; por isso, apresentaremos erros **na escala logarítmica e na escala original**. Não estamos treinando um classificador de disfunção nem prevendo diretamente o atendimento à regra de prioridade.

A cópia do escore da origem é uma referência essencial: o modelo precisa acrescentar informação além de simplesmente esperar que a próxima réplica seja parecida. Comparamos mediana do treinamento, cópia, calibração linear, Ridge com quatro escores, Ridge com medidas e XGBoost.

Todos os registros de um mesmo composto, inclusive doses e placas diferentes, ficam no mesmo fold externo. O conjunto externo não escolhe características, parâmetros ou número de árvores. A parada antecipada usa apenas compostos internos separados. **O alvo e a coorte, entretanto, já foram definidos no desenvolvimento completo.** A validação é condicional a essa definição, não uma validação independente de todo o pipeline. Anotações farmacológicas e R4 não entram no modelo.

Se houver poucos compostos para as partições configuradas, somente a modelagem será marcada como não executada; a descoberta, a anotação e as tabelas anteriores continuam disponíveis.

In [ ]:
MODELOS = ['Mediana_treino', 'Copiar_escore', 'Calibracao_escore', 'Ridge_4_escores', 'Ridge_medidas', 'XGBoost']
INDICES_X = np.flatnonzero(SEMANTICA.uso.ne('excluir').to_numpy() & escala_ok)
NOMES_X = [NOMES[i] for i in INDICES_X]
SEM_X = SEMANTICA.iloc[INDICES_X].reset_index(drop=True)
X_CUBO = agregar_matriz(meta, Z[:, INDICES_X], NOMES_X, META_COMPLETO, REPS)

def selecionar_no_treino(x, sem):
    finito = np.isfinite(x)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        med = np.nanmedian(np.where(finito, x, np.nan), axis=0)
        iqr = np.nanquantile(np.where(finito, x, np.nan), .75, axis=0) - np.nanquantile(np.where(finito, x, np.nan), .25, axis=0)
    bons = np.flatnonzero((finito.mean(0) >= CFG['min_fracao_finita_X']) & np.isfinite(med) & (iqr > CFG['min_IQR_X']))
    exigir(len(bons) > 0, 'Sem preditores utilizaveis no treinamento.')
    preenchido = np.where(finito, x, med)
    escolhidos = []
    for _, sub in sem.iloc[bons].groupby(['bloco', 'familia'], sort=True):
        ids = sub.index.to_numpy()
        ranks = rankdata(preenchido[:, ids], axis=0)
        ranks -= ranks.mean(0)
        ranks /= np.linalg.norm(ranks, axis=0)
        corr = ranks.T @ ranks
        aceitos = []
        for j, indice in enumerate(ids):
            if not any(abs(corr[j, k]) >= CFG['correlacao_maxima'] for k in aceitos):
                aceitos.append(j)
                escolhidos.append(indice)
    ids = np.asarray(sorted(escolhidos), dtype=int)
    return {'indices': ids, 'medianas': med[ids], 'IQR': iqr[ids]}

def transformar_x(x, pre, escalar=False):
    v = x[:, pre['indices']]
    v = np.where(np.isfinite(v), v, pre['medianas'])
    return (v - pre['medianas']) / pre['IQR'] if escalar else v

def criar_xgb(n_arvores, seed, paciencia=None):
    extras = {'early_stopping_rounds': paciencia} if paciencia else {}
    return XGBRegressor(n_estimators=int(n_arvores), random_state=seed, **CFG['xgb'], **extras)

def metricas(y, pred):
    def r2(a, b):
        den = np.sum((a - a.mean()) ** 2)
        return 1 - np.sum((a - b) ** 2) / den if den > 0 else np.nan
    with np.errstate(over='ignore', invalid='ignore'):
        a, p = np.expm1(y), np.expm1(pred)
        return {'MAE_log': float(np.mean(np.abs(y - pred))),
                'RMSE_log': float(np.sqrt(np.mean((y - pred) ** 2))), 'R2_log': float(r2(y, pred)),
                'Spearman': float(spearmanr(y, pred).statistic) if np.ptp(pred) > 0 else np.nan,
                'MAE_A': float(np.mean(np.abs(a - p))),
                'RMSE_A': float(np.sqrt(np.mean((a - p) ** 2))), 'R2_A': float(r2(a, p)),
                'n_previsoes_A_negativas': int((p < 0).sum()), 'n_previsoes_A_nao_finitas': int((~np.isfinite(p)).sum())}

PODE_MODELAR = (CFG['executar_modelos'] and META_COMPLETO.composto.nunique() >= 2 * CFG['folds_externos'])
if not PODE_MODELAR:
    AVISOS.append('Modelagem nao executada: desativada ou poucos compostos para a separacao configurada.')

## 15. Treinar os comparadores e explicar somente as predições externas

A cada direção, o modelo recebe características da origem e tenta prever o escore no destino. As mesmas divisões de compostos são usadas em todas as direções e comparadores. O pré-processamento de X é aprendido novamente dentro do treinamento interno e externo; as medidas não são escolhidas usando os exemplos que depois avaliamos.

**SHAP** decompõe a saída do modelo em um valor de referência e contribuições por característica. Um valor positivo aumenta a previsão de `log(1+A_alvo)` em relação à referência daquele modelo. Não significa que a característica aumentou, que a mitocôndria foi ativada ou que se demonstrou causalidade. A identidade `base + soma(SHAP) = previsão` é conferida em todos os exemplos externos.

Também reconstruímos o Ridge pelos coeficientes e entradas escaladas. Valores extremos ou negativos não são truncados. A exponencial pode ampliar muito um erro logarítmico; essa é uma propriedade da transformação, não um motivo para esconder casos.

In [ ]:
PREDICOES, AUDITORIA_MODELOS, IMPORTANCIAS, SHAP_GRUPOS, TERMOS_EXTREMOS = [], [], [], [], []
if PODE_MODELAR:
    splitter = StratifiedGroupKFold(n_splits=CFG['folds_externos'], shuffle=True, random_state=CFG['seed'])
    folds = list(splitter.split(np.zeros(len(META_COMPLETO)), META_COMPLETO.placa, groups=META_COMPLETO.composto))
    mapa_folds = META_COMPLETO[['id_condicao', 'composto', 'placa', 'dose']].copy()
    mapa_folds['fold'] = -1
    for fold, (_, teste) in enumerate(folds): mapa_folds.loc[teste, 'fold'] = fold + 1
    exigir(mapa_folds.groupby('composto').fold.nunique().eq(1).all(), 'Composto dividido entre folds.')
    tabela('11_particoes_compostos', mapa_folds)
    for origem, destino in permutations(range(len(REPS)), 2):
        direcao = REPS[origem] + '_para_' + REPS[destino]
        x, y = X_CUBO[:, origem], np.log1p(A_COMPLETO[:, destino, J_ALVO])
        escore_origem = np.log1p(A_COMPLETO[:, origem, J_ALVO])
        quatro = np.log1p(A_COMPLETO[:, origem, :len(CFG['canais'])])
        for fold, (treino, teste) in enumerate(folds, 1):
            tag = direcao + '_fold' + str(fold)
            cp = SAIDA / 'checkpoints' / (tag + '.json')
            arquivos = {'pre': SAIDA / 'modelos' / (tag + '_pre.json'),
                        'linear': SAIDA / 'modelos' / (tag + '_lineares.json'),
                        'modelo': SAIDA / 'modelos' / (tag + '.ubj'),
                        'shap': SAIDA / 'shap' / (tag + '.npz'),
                        'pred': SAIDA / 'checkpoints' / (tag + '_pred.csv')}
            exigir(not set(META_COMPLETO.iloc[treino].composto) & set(META_COMPLETO.iloc[teste].composto),
                   'Vazamento de composto entre treino e avaliacao.')
            if cp.exists():
                salvo = json.loads(cp.read_text())
                exigir(salvo['protocolo'] == HASH_PROTOCOLO and salvo['descoberta'] == HASH_DESCOBERTA,
                       'Checkpoint pertence a outra analise.')
                for nome, path in arquivos.items(): exigir(sha256(path) == salvo['hashes'][nome], 'Checkpoint alterado: ' + str(path))
                pre = json.loads(arquivos['pre'].read_text())
                for nome in ['indices', 'medianas', 'IQR']: pre[nome] = np.asarray(pre[nome])
                lineares = json.loads(arquivos['linear'].read_text())
                predsalva = pd.read_csv(arquivos['pred'])
                exigir(np.array_equal(predsalva.indice, teste), 'Ordem de checkpoint diferente.')
                preds = {nome: predsalva[nome].to_numpy() for nome in MODELOS}
                with np.load(arquivos['shap'], allow_pickle=False) as dados:
                    sv, xe, base = dados['values'], dados['X'], float(dados['base'])
                auditoria = salvo['auditoria']
            else:
                interno = StratifiedGroupKFold(n_splits=CFG['folds_internos'], shuffle=True, random_state=CFG['seed'])
                it, iv = next(interno.split(np.zeros(len(treino)), META_COMPLETO.iloc[treino].placa,
                                            groups=META_COMPLETO.iloc[treino].composto))
                ti, vi = treino[it], treino[iv]
                exigir(not set(META_COMPLETO.iloc[ti].composto) & set(META_COMPLETO.iloc[vi].composto), 'Vazamento interno.')
                pre_interno = selecionar_no_treino(x[ti], SEM_X)
                pre = selecionar_no_treino(x[treino], SEM_X)
                xi, xv = transformar_x(x[ti], pre_interno), transformar_x(x[vi], pre_interno)
                xt, xe = transformar_x(x[treino], pre), transformar_x(x[teste], pre)
                si, svalid = transformar_x(x[ti], pre_interno, True), transformar_x(x[vi], pre_interno, True)
                st, se = transformar_x(x[treino], pre, True), transformar_x(x[teste], pre, True)
                seed = CFG['seed'] + fold
                with threadpool_limits(limits=2):
                    cal = LinearRegression().fit(escore_origem[treino, None], y[treino])
                    ridge4 = Ridge(alpha=1.0).fit(quatro[treino], y[treino])
                    escolhas = []
                    for alpha in CFG['ridge_alphas']:
                        r = Ridge(alpha=alpha).fit(si, y[ti])
                        escolhas.append((float(np.mean(np.abs(y[vi] - r.predict(svalid)))), alpha))
                    alpha = min(escolhas)[1]
                    ridge = Ridge(alpha=alpha).fit(st, y[treino])
                    provis = criar_xgb(CFG['max_arvores'], seed, CFG['paciencia'])
                    provis.fit(xi, y[ti], eval_set=[(xv, y[vi])], verbose=False)
                    n_arvores = int(provis.best_iteration) + 1
                    modelo = criar_xgb(n_arvores, seed)
                    modelo.fit(xt, y[treino], verbose=False)
                    preds = {'Mediana_treino': np.full(len(teste), np.median(y[treino])),
                             'Copiar_escore': escore_origem[teste],
                             'Calibracao_escore': cal.predict(escore_origem[teste, None]),
                             'Ridge_4_escores': ridge4.predict(quatro[teste]),
                             'Ridge_medidas': ridge.predict(se),
                             'XGBoost': modelo.predict(xe).astype(float)}
                    explicador = shap.TreeExplainer(modelo, feature_perturbation='tree_path_dependent', model_output='raw')
                    sv = np.asarray(explicador.shap_values(xe), dtype=float)
                    base = float(np.asarray(explicador.expected_value).ravel()[0])
                lineares = {'coef': ridge.coef_.tolist(), 'intercepto': float(ridge.intercept_),
                            'alpha': float(alpha), 'min_y_treino': float(y[treino].min()),
                            'max_y_treino': float(y[treino].max())}
                auditoria = {'modelo_fold': tag, 'n_treino': len(treino), 'n_teste': len(teste),
                            'n_preditores': len(pre['indices']), 'n_arvores': n_arvores,
                            'alpha_Ridge': float(alpha), 'n_imputacoes_externas': int((~np.isfinite(x[teste][:, pre['indices']])).sum())}
                modelo.save_model(arquivos['modelo'])
                gravar_json(arquivos['pre'], {**pre, 'nomes': [NOMES_X[i] for i in pre['indices']]})
                gravar_json(arquivos['linear'], lineares)
                np.savez_compressed(arquivos['shap'], values=sv, X=xe, base=base,
                                    indices=teste, ids_condicao=META_COMPLETO.iloc[teste].id_condicao.to_numpy(),
                                    nomes=np.asarray([NOMES_X[i] for i in pre['indices']]), y=y[teste], prediction=preds['XGBoost'])
                pd.DataFrame({'indice': teste, **preds}).to_csv(arquivos['pred'], index=False)
            erro_shap = float(np.max(np.abs(base + sv.sum(1) - preds['XGBoost'])))
            exigir(erro_shap < 2e-4, 'SHAP nao aditivo: ' + tag)
            escalado = (xe - pre['medianas']) / pre['IQR']
            termos = escalado * np.asarray(lineares['coef'])
            reconstruida = lineares['intercepto'] + termos.sum(1)
            erro_ridge = float(np.max(np.abs(reconstruida - preds['Ridge_medidas'])))
            exigir(np.allclose(reconstruida, preds['Ridge_medidas'], rtol=1e-8, atol=1e-8), 'Ridge nao reproduzido: ' + tag)
            auditoria.update({'erro_aditivo_SHAP': erro_shap, 'erro_reconstrucao_Ridge': erro_ridge})
            AUDITORIA_MODELOS.append(auditoria)
            for nome, pred in preds.items():
                bloco = META_COMPLETO.iloc[teste][['id_condicao'] + CHAVE + ['dose']].copy()
                bloco['direcao'], bloco['origem'], bloco['destino'] = direcao, REPS[origem], REPS[destino]
                bloco['fold'], bloco['modelo'] = fold, nome
                bloco['y_log'], bloco['pred_log'] = y[teste], pred
                bloco['A_observado'] = A_COMPLETO[teste, destino, J_ALVO]
                with np.errstate(over='ignore'): bloco['A_predito'] = np.expm1(pred)
                bloco['A_origem'] = A_COMPLETO[teste, origem, J_ALVO]
                PREDICOES.append(bloco)
            nomes_ajuste = [NOMES_X[i] for i in pre['indices']]
            imp = SEM_X.iloc[pre['indices']].copy()
            imp['modelo_fold'], imp['media_abs_SHAP'], imp['n_externos'] = tag, np.abs(sv).mean(0), len(teste)
            IMPORTANCIAS.append(imp)
            for campo in ['bloco', 'familia']:
                for grupo, linhas in imp.groupby(campo):
                    ids = np.flatnonzero(imp[campo].to_numpy() == grupo)
                    resumo = META_COMPLETO.iloc[teste][['id_condicao', 'composto']].copy()
                    resumo['direcao'], resumo['modelo_fold'], resumo['tipo'], resumo['grupo'] = direcao, tag, campo, grupo
                    resumo['SHAP_absoluto'], resumo['SHAP_sinal'] = np.abs(sv[:, ids]).sum(1), sv[:, ids].sum(1)
                    resumo['n_medidas'] = len(ids)
                    SHAP_GRUPOS.append(resumo)
            # Decompor extremos em ambas as escalas; nao excluir qualquer observacao.
            with np.errstate(over='ignore', invalid='ignore'):
                erro_a = np.abs(np.expm1(preds['Ridge_medidas']) - np.expm1(y[teste]))
            extremos = set(np.argsort(-erro_a)[:3]) | {int(np.argmax(np.abs(y[teste] - preds['Ridge_medidas'])))}
            for i in sorted(extremos):
                for j in np.argsort(-np.abs(termos[i]))[:20]:
                    TERMOS_EXTREMOS.append({'modelo_fold': tag, 'id_condicao': int(META_COMPLETO.iloc[teste[i]].id_condicao),
                        'composto': META_COMPLETO.iloc[teste[i]].composto, 'medida': nomes_ajuste[j],
                        'valor_MAD': xe[i, j], 'mediana_treino': pre['medianas'][j], 'IQR_treino': pre['IQR'][j],
                        'coeficiente': lineares['coef'][j], 'termo_linear': termos[i, j],
                        'pred_log': reconstruida[i], 'observado_log': y[teste[i]]})
            if not cp.exists():
                gravar_json(cp, {'protocolo': HASH_PROTOCOLO, 'descoberta': HASH_DESCOBERTA,
                                 'hashes': {n: sha256(p) for n, p in arquivos.items()}, 'auditoria': auditoria})
            print(tag, '| variaveis:', auditoria['n_preditores'], '| arvores:', auditoria['n_arvores'])

## 16. Comparar erros, examinar extremos e ler SHAP

As métricas são calculadas por direção e depois resumidas com peso igual. **R² não é porcentagem de acerto**. Um modelo pode ter menor erro em log e não ter menor erro na escala A, especialmente se houver valores extremos. As figuras mantêm todos os pontos e a tabela de extremos mostra contagens de objetos e dominância de medidas na origem e no destino.

O bootstrap compara os erros reamostrando **compostos**, mantendo juntas doses e direções. Suas faixas são condicionais às predições já salvas, não a novos experimentos. O resumo SHAP combina diferentes modelos e referências; medidas ausentes de um ajuste recebem contribuição zero na média global. A frequência de seleção é apresentada.

O caderno conserva as matrizes SHAP completas em `shap/`, com nomes das medidas, entradas e referência de cada modelo. As 20 características mostradas no gráfico global são apenas uma seleção para apresentação; as matrizes completas permanecem disponíveis. Nenhuma observação foi escolhida por uma explicação favorável.

In [ ]:
METRICAS, SHAP_GLOBAL = pd.DataFrame(), pd.DataFrame()
PRED = pd.DataFrame()
if PREDICOES:
    PRED = pd.concat(PREDICOES, ignore_index=True)
    exigir(not PRED.duplicated(['id_condicao', 'direcao', 'modelo']).any(), 'Predicoes duplicadas.')
    n_direcoes = len(REPS) * (len(REPS) - 1)
    exigir(len(PRED) == len(META_COMPLETO) * n_direcoes * len(MODELOS), 'Predicoes incompletas.')
    PRED['erro_abs_log'] = (PRED.pred_log - PRED.y_log).abs()
    PRED['erro_abs_A'] = (PRED.A_predito - PRED.A_observado).abs()
    tabela('12_predicoes_externas', PRED)
    tabela('12_auditoria_modelos', pd.DataFrame(AUDITORIA_MODELOS))
    tabela('12_Ridge_termos_extremos', pd.DataFrame(TERMOS_EXTREMOS))
    por_direcao = pd.DataFrame([{'direcao': direcao, 'modelo': modelo, **metricas(q.y_log.to_numpy(), q.pred_log.to_numpy())}
        for (direcao, modelo), q in PRED.groupby(['direcao', 'modelo'])])
    tabela('12_metricas_direcao', por_direcao)
    METRICAS = tabela('12_metricas_resumo', por_direcao.groupby('modelo').agg(
        {c: lambda s: s.mean(skipna=False) for c in ['MAE_log', 'RMSE_log', 'R2_log', 'Spearman', 'MAE_A', 'RMSE_A', 'R2_A']}).reset_index())
    tabela('12_metricas_placa', pd.DataFrame([{'placa': p, 'direcao': d, 'modelo': m,
        **metricas(q.y_log.to_numpy(), q.pred_log.to_numpy())} for (p, d, m), q in PRED.groupby(['placa', 'direcao', 'modelo'])]))
    erros_compostos = PRED.groupby(['composto', 'modelo']).erro_abs_log.mean().unstack()
    rng = np.random.default_rng(CFG['seed'])
    comparacoes = []
    for comparador in [m for m in MODELOS if m != 'XGBoost']:
        delta = (erros_compostos.XGBoost - erros_compostos[comparador]).to_numpy()
        sorteios = [delta[rng.integers(0, len(delta), len(delta))].mean() for _ in range(CFG['bootstrap_modelos'])]
        comparacoes.append({'comparador': comparador, 'delta_MAE_log': delta.mean(),
                           'percentil_2_5': np.quantile(sorteios, .025), 'percentil_97_5': np.quantile(sorteios, .975)})
    tabela('12_bootstrap_pareado_compostos', pd.DataFrame(comparacoes))
    extremos = []
    for (d, m), sub in PRED.groupby(['direcao', 'modelo']):
        for escala in ['log', 'A']:
            q = sub.nlargest(10, 'erro_abs_' + escala).copy()
            q['escala_ordenacao'] = escala
            extremos.append(q)
    extremos = pd.concat(extremos, ignore_index=True)
    diag = DOMINANCIA[DOMINANCIA.bloco == ALVO].merge(meta[['placa', 'replica', 'poco', 'dose_chave']], on=['placa', 'replica', 'poco'], validate='one_to_one')
    diag = diag.groupby(CHAVE + ['replica']).agg(n_objetos_min=('n_objetos', 'min'),
        maior_dominancia=('fracao_quadratica_maior_medida', 'max')).reset_index()
    for lado in ['origem', 'destino']:
        novo = diag.rename(columns={'replica': lado, 'n_objetos_min': 'n_objetos_' + lado,
                                    'maior_dominancia': 'dominancia_' + lado})
        extremos = extremos.merge(novo, on=CHAVE + [lado], how='left', validate='many_to_one')
    tabela('12_extremos_sem_exclusao', extremos)
    imp = pd.concat(IMPORTANCIAS, ignore_index=True)
    tabela('13_SHAP_por_modelo', imp)
    imp['soma_abs'] = imp.media_abs_SHAP * imp.n_externos
    SHAP_GLOBAL = imp.groupby('medida').agg(soma_abs=('soma_abs', 'sum'), n_modelos=('modelo_fold', 'nunique')).reset_index()
    SHAP_GLOBAL = SEM_X.merge(SHAP_GLOBAL, on='medida', how='left').fillna({'soma_abs': 0, 'n_modelos': 0})
    SHAP_GLOBAL['SHAP_abs_medio'] = SHAP_GLOBAL.soma_abs / (len(META_COMPLETO) * n_direcoes)
    SHAP_GLOBAL['frequencia_selecao'] = SHAP_GLOBAL.n_modelos / len(AUDITORIA_MODELOS)
    SHAP_GLOBAL = SHAP_GLOBAL.sort_values('SHAP_abs_medio', ascending=False)
    tabela('13_SHAP_global', SHAP_GLOBAL)
    GRUPOS_SHAP = pd.concat(SHAP_GRUPOS, ignore_index=True)
    tabela('13_SHAP_grupos_com_sinal', GRUPOS_SHAP)
    fig, ax = plt.subplots(figsize=(8, 4))
    q = METRICAS.sort_values('MAE_log')
    ax.barh(q.modelo, q.MAE_log)
    ax.set(xlabel='MAE em log(1+A)', title='Comparacao externa entre modelos')
    ax.invert_yaxis()
    figura('03_modelos', fig, 'Media com peso igual das direcoes. Os modelos sao avaliados em codigos omitidos do ajuste, condicionados ao alvo anterior.')
    fig, ax = plt.subplots(figsize=(6, 6))
    q = PRED[PRED.modelo == 'XGBoost']
    ax.scatter(q.y_log, q.pred_log, s=7, alpha=.25)
    limites = [min(q.y_log.min(), q.pred_log.min()), max(q.y_log.max(), q.pred_log.max())]
    ax.plot(limites, limites, linestyle='--')
    ax.set(xlabel='Observado log(1+A)', ylabel='Previsto log(1+A)', title='XGBoost: todos os exemplos externos')
    figura('04_observado_previsto', fig, 'Todas as direcoes reutilizam condicoes; nao sao experimentos independentes. Nenhum extremo foi excluido.')
    fig, ax = plt.subplots(figsize=(10, 7))
    q = SHAP_GLOBAL.head(20).iloc[::-1]
    ax.barh(q.medida, q.SHAP_abs_medio)
    ax.set(xlabel='Media de |SHAP| em log(1+A)', title='Contribuicoes preditivas globais')
    figura('05_SHAP_global', fig, 'Resumo de modelos externos diferentes. Contribuicao nao indica causalidade ou efeito biologico.')
    display(METRICAS)

## 17. Identificar os compostos depois da análise

A ponte documental procura a identidade de **todos os códigos desta execução**. Para a biblioteca EU-OS, utiliza as tabelas `EOS -> pdid -> identidade` e a tabela longa de alvos. Para outra biblioteca, forneça `IDENTIDADES_CSV` com as colunas `composto` e `name`, e opcionalmente `smiles` e `inchikey`. Registros de alvos podem ser fornecidos separadamente em `ALVOS_CSV`, com uma coluna `composto`.

Correspondências ausentes não removem candidatos. O resultado fica como `sem_correspondencia` ou `sem_anotacao`. Nomes parecidos não são usados para inventar identidade. As estruturas são conferidas sem retirar sais, neutralizar cargas ou padronizar tautomeria. Uma discordância fica sinalizada; não altera o modelo ou o candidato.

Ações, alvos e espécies permanecem relacionados nas linhas originais. Uma lista resumida de genes e uma lista de ações não permitem combinar todos os genes com todas as ações. As anotações são contexto documental, não evidência de que o alvo causou o perfil observado.

In [ ]:
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.warning')
M_LINK = 'annotations/2024-08-02_EOS_pdid.csv'
M_ID = 'annotations/pd_export_04_2022_2464_compounds_standardized.csv'
M_TARGET = 'annotations/pd_export_04_2022_2464_targets_standardized.csv'
identidade_fonte = pd.DataFrame(columns=['composto', 'name', 'smiles', 'inchikey'])
alvos = pd.DataFrame(columns=['composto'])
if IDENTIDADES_CSV:
    identidade_fonte = ler_documento_csv(IDENTIDADES_CSV).rename(columns=COLUNAS)
    exigir({'composto', 'name'} <= set(identidade_fonte), 'Identidades: faltam composto/name.')
    identidade_fonte['fonte_identidade'] = str(IDENTIDADES_CSV)
elif FONTES_ZIP and Path(FONTES_ZIP).is_file():
    with zipfile.ZipFile(FONTES_ZIP) as z: membros = set(z.namelist())
    if {M_LINK, M_ID, M_TARGET} <= membros:
        link = ler_documento_csv(FONTES_ZIP, M_LINK).rename(columns={'Metadata_EOS': 'composto', 'Metadata_pdid': 'pdid'})
        ids = ler_documento_csv(FONTES_ZIP, M_ID)
        exigir(link.composto.is_unique and ids.pdid.is_unique and link.pdid.is_unique, 'Ponte documental ambigua.')
        identidade_fonte = link.merge(ids, on='pdid', how='left', validate='one_to_one')
        identidade_fonte['fonte_identidade'] = M_ID
        alvos = ler_documento_csv(FONTES_ZIP, M_TARGET)
        alvos['linha_fonte'] = np.arange(2, len(alvos) + 2)
        alvos['fonte_alvo'] = M_TARGET
        alvos = alvos.merge(link[['pdid', 'composto']], on='pdid', how='inner', validate='many_to_one')
    else:
        AVISOS.append('ZIP sem tabelas de identidades: a descoberta foi mantida sem anotacoes.')
if ALVOS_CSV:
    alvos = ler_documento_csv(ALVOS_CSV).rename(columns=COLUNAS)
    exigir('composto' in alvos, 'A tabela de alvos exige a coluna composto.')
    alvos['fonte_alvo'] = str(ALVOS_CSV)
exigir(identidade_fonte.composto.is_unique, 'Identidades duplicadas: nao escolher uma arbitrariamente.')
for col in ['name', 'smiles', 'inchikey']:
    if col not in identidade_fonte: identidade_fonte[col] = ''
IDENTIDADES = pd.DataFrame({'composto': sorted(condicoes.composto.unique())}).merge(
    identidade_fonte, on='composto', how='left', validate='one_to_one', indicator=True)
IDENTIDADES['status_identidade'] = np.where(IDENTIDADES['_merge'].eq('both'), 'codigo_exato', 'sem_correspondencia')
IDENTIDADES = IDENTIDADES.drop(columns='_merge').fillna('')
canonicos, estruturas = [], []
for row in IDENTIDADES.itertuples():
    mol = Chem.MolFromSmiles(row.smiles) if row.smiles else None
    if not row.smiles: can, estado = '', 'estrutura_nao_fornecida'
    elif mol is None: can, estado = '', 'SMILES_invalido'
    else:
        can = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        estado = ('concordante' if Chem.MolToInchiKey(mol) == row.inchikey else 'discordante') if row.inchikey else 'sem_InChIKey_referencia'
    canonicos.append(can)
    estruturas.append(estado)
IDENTIDADES['SMILES_canonico'] = canonicos
IDENTIDADES['status_estrutura'] = estruturas
alvos = alvos[alvos.composto.isin(IDENTIDADES.composto)].copy()
contagens_alvos = alvos.groupby('composto').size()
IDENTIDADES['n_registros_alvos'] = IDENTIDADES.composto.map(contagens_alvos).fillna(0).astype(int)
IDENTIDADES['farmacologia_utilizavel'] = IDENTIDADES.status_estrutura.eq('concordante') & IDENTIDADES.n_registros_alvos.gt(0)
for coluna in ['gene_name', 'moa']:
    if coluna in alvos:
        valores = alvos.groupby('composto')[coluna].agg(lambda s: ' | '.join(sorted(set(s.dropna().astype(str)) - {''})))
        IDENTIDADES[coluna + '_resumo'] = IDENTIDADES.composto.map(valores).fillna('')
tabela('14_identidades_toda_coorte', IDENTIDADES)
tabela('14_alvos_documentais_longos', alvos)
tabela('14_identidades_para_revisao', IDENTIDADES[~IDENTIDADES.status_estrutura.eq('concordante')])
CANDIDATOS_IDENTIFICADOS = candidatos.merge(IDENTIDADES, on='composto', how='left', validate='many_to_one')
exigir(len(CANDIDATOS_IDENTIFICADOS) == len(candidatos), 'Anotacao alterou a lista de candidatos.')
tabela('14_candidatos_identificados', CANDIDATOS_IDENTIFICADOS)
COBERTURA = tabela('14_cobertura', pd.DataFrame([
    {'indicador': 'codigos_na_coorte', 'n': len(IDENTIDADES)},
    {'indicador': 'correspondencia_exata', 'n': int(IDENTIDADES.status_identidade.eq('codigo_exato').sum())},
    {'indicador': 'sem_correspondencia', 'n': int(IDENTIDADES.status_identidade.eq('sem_correspondencia').sum())},
    {'indicador': 'estrutura_concordante', 'n': int(IDENTIDADES.status_estrutura.eq('concordante').sum())},
    {'indicador': 'com_registros_de_alvos', 'n': int(IDENTIDADES.n_registros_alvos.gt(0).sum())}]))
if len(PRED):
    xp = PRED[PRED.modelo == 'XGBoost'].merge(IDENTIDADES, on='composto', how='left', validate='many_to_one')
    exigir(len(xp) == len(META_COMPLETO) * len(REPS) * (len(REPS)-1), 'Predicoes duplicadas apos ponte.')
    tabela('14_XGBoost_com_identidades', xp)
    # Primeiro direcoes dentro da condicao; depois condicoes dentro do composto.
    sg = GRUPOS_SHAP.groupby(['id_condicao', 'composto', 'tipo', 'grupo']).agg(
        SHAP_abs=('SHAP_absoluto', 'mean'), SHAP_sinal=('SHAP_sinal', 'mean')).reset_index()
    sg = sg.groupby(['composto', 'tipo', 'grupo'])[['SHAP_abs', 'SHAP_sinal']].mean().reset_index()
    tabela('14_SHAP_por_composto', sg.merge(IDENTIDADES, on='composto', how='left', validate='many_to_one'))
    if 'ChEMBL [LEVEL 1]' in alvos:
        membros = alvos.loc[alvos['ChEMBL [LEVEL 1]'].ne(''), ['composto', 'ChEMBL [LEVEL 1]']].drop_duplicates()
        membros = membros[membros.composto.isin(IDENTIDADES.loc[IDENTIDADES.farmacologia_utilizavel, 'composto'])]
        categorias = membros.merge(sg, on='composto', how='inner')
        tabela('14_SHAP_categorias_descritivo', categorias.groupby(['ChEMBL [LEVEL 1]', 'tipo', 'grupo']).agg(
            n_compostos=('composto', 'nunique'), SHAP_abs_medio=('SHAP_abs', 'mean'), SHAP_sinal_medio=('SHAP_sinal', 'mean')).reset_index())
display(COBERTURA)
display(CANDIDATOS_IDENTIFICADOS[['composto', 'name', 'placa', 'dose', 'status_identidade', 'status_estrutura']])

## 18. Abrir a réplica reservada: avaliação, não uma nova busca de hits

A descoberta já percorreu toda a coorte de desenvolvimento. Agora avaliamos **a lista encontrada nesta execução**, seja qual for seu tamanho. Não se procura sempre pelos mesmos três nomes; procura-se pela mesma condição dos candidatos recém-encontrados. Os parâmetros do alvo e o P95 permanecem congelados.

Por segurança, a abertura está desativada. A primeira execução pode terminar com toda a análise de desenvolvimento concluída. Depois de revisar o protocolo e a lista, altere **somente** `ABRIR_RESERVADA` para `True` nesta célula e execute esta seção e o fechamento, sem repetir o treinamento. Se `CFG['reservada'] = None`, não existe esta etapa. Para usar todas as réplicas na descoberta, declare-as no desenvolvimento, retire a reserva e registre o resultado como **exploratório sem teste reservado**.

No FMP atual, R4 já foi examinada. Esta é uma reanálise, não uma nova validação intocada. Para novos dados realmente não examinados, preencha esse estado corretamente na ficha inicial.

Os DMSO da reserva normalizam apenas a própria réplica. MAD inválido, medida ausente ou condição sem perfil produzem `nao_avaliavel`, não um resultado negativo. A auditoria identifica o nome exato da medida que inviabilizou o bloco, sem removê-la após olhar R4. Nenhum modelo é treinado na reserva.

**Nesta reanálise, R4 já foi consultada no estudo anterior.** Mantenha `reserva_ja_consultada=True`. Para incluir a avaliação retrospectiva, altere somente `ABRIR_RESERVADA` nesta seção, execute-a e depois execute novamente a seção 19. Não mude os critérios após examinar essa avaliação.


In [ ]:
ABRIR_RESERVADA = False
AVALIACAO_RESERVA = pd.DataFrame()
AUD_RESERVA = pd.DataFrame()
if ABRIR_RESERVADA and CFG['reservada'] is not None:
    exigir(bool(RES_FILES), 'Nao ha arquivos da replica reservada declarada.')
    exigir(sha256(SAIDA / 'protocolo.json') == HASH_PROTOCOLO, 'Protocolo alterado.')
    exigir(CFG == PROTOCOLO['config'] and COLUNAS == PROTOCOLO['colunas'], 'Configuracao alterada em memoria.')
    exigir(sha256(SAIDA / 'descoberta_congelada.json') == HASH_DESCOBERTA, 'Descoberta alterada.')
    for path, h in ORIGINAIS.items(): exigir(sha256(path) == h, 'Fonte alterada: ' + path)
    registro = SAIDA / 'reserva_aberta.json'
    if not registro.exists():
        gravar_json(registro, {'utc': datetime.now(timezone.utc).isoformat(),
            'replica': CFG['reservada'], 'ja_consultada_antes_desta_execucao': CFG['reserva_ja_consultada'],
            'descoberta_sha256': HASH_DESCOBERTA})
    meta_res, X_res = ler_valores(RES_FILES, NOMES)
    if CFG['min_objetos_inclusao'] is not None:
        exigir(meta_res.n_objetos.notna().all(), 'Contagem ausente na reserva.')
        incluir = meta_res.n_objetos.ge(CFG['min_objetos_inclusao']).to_numpy()
        tabela('15_exclusoes_reserva_contagem', meta_res.loc[~incluir])
        meta_res, X_res = meta_res.loc[incluir].reset_index(drop=True), X_res[incluir]
    exigir(len(meta_res) > 0, 'Nenhum perfil restante na reserva.')
    exigir(not np.isinf(X_res).any(), 'Valores brutos infinitos na reserva; verifique a entrada.')
    Z_res, parametros_res = normalizar_dmsos(meta_res, X_res, NOMES)
    tabela('15_parametros_reserva', parametros_res)
    invalidos = parametros_res.merge(CATALOGO[['medida', 'bloco', 'selecionada']], on='medida', validate='many_to_one')
    tabela('15_medidas_selecionadas_invalidas', invalidos[invalidos.selecionada & ~invalidos.escala_valida])
    scores_res, _, dom_res = escores(meta_res, Z_res, REFERENCIAS)
    AUD_RESERVA = auditar_controles(scores_res)
    tabela('15_escores_reserva', scores_res)
    tabela('15_dominancia_reserva', dom_res)
    tabela('15_DMSO_reserva', AUD_RESERVA)
    ar = agregar_matriz(meta_res, scores_res[COL_A].to_numpy(), COL_A, condicoes, [CFG['reservada']])[:, 0]
    zr = agregar_matriz(meta_res, Z_res[:, INDICES_BLOCO[ALVO]], NOMES_ALVO, condicoes, [CFG['reservada']])
    linhas = []
    for row in candidatos.itertuples():
        i = row.id_condicao
        cos = np.asarray([cosseno(zr[i, 0], z) for z in Z_ALVO[i]])
        avaliavel = np.isfinite(ar[i]).all() and np.isfinite(cos).all()
        passou_alvo = ar[i, J_ALVO] > 1
        passou_outros = (ar[i, OUTROS] <= 1).all()
        passou_cosseno = (cos > CFG['cosseno_minimo']).all()
        estado = 'nao_avaliavel' if not avaliavel else ('padrao_repetido' if passou_alvo and passou_outros and passou_cosseno else 'padrao_nao_repetido')
        registro_cand = {'id_condicao': i, 'composto': row.composto, 'placa': row.placa, 'dose': row.dose,
            'estado_reserva': estado, 'A_alvo_desenvolvimento': np.median(A_CUBO[i, :, J_ALVO]),
            'A_alvo_reserva': ar[i, J_ALVO], 'cosseno_min_reserva': float(np.min(cos)),
            'motivos': 'medidas_escalas_ou_perfis_nao_avaliaveis' if not avaliavel else ' | '.join(
                m for m, p in [('magnitude_alvo', passou_alvo), ('outros_blocos', passou_outros), ('direcao', passou_cosseno)] if not p)}
        registro_cand.update({'A_' + b + '_reserva': ar[i, j] for j, b in enumerate(BLOCOS)})
        linhas.append(registro_cand)
    AVALIACAO_RESERVA = pd.DataFrame(linhas, columns=['id_condicao', 'composto', 'placa', 'dose',
        'estado_reserva', 'A_alvo_desenvolvimento', 'A_alvo_reserva', 'cosseno_min_reserva', 'motivos'] + ['A_' + b + '_reserva' for b in BLOCOS])
    AVALIACAO_RESERVA = AVALIACAO_RESERVA.merge(IDENTIDADES, on='composto', how='left', validate='many_to_one')
    tabela('15_avaliacao_candidatos_reserva', AVALIACAO_RESERVA)
    # Contribuicoes da reserva e das mesmas medidas top20 escolhidas no desenvolvimento.
    cr, fr = caracterizar(candidatos, zr, [CFG['reservada']], '15_assinaturas_reserva')
    if len(cr):
        ordem_dev = CARACTERISTICAS_CANDIDATOS[['id_condicao', 'medida', 'ordem_exibicao']].rename(columns={'ordem_exibicao': 'ordem_desenvolvimento'})
        cr = cr.merge(ordem_dev, on=['id_condicao', 'medida'], validate='one_to_one')
        tabela('15_top20_desenvolvimento_acompanhado', cr[cr.ordem_desenvolvimento <= 20])
    display(AVALIACAO_RESERVA[['composto', 'name', 'A_alvo_desenvolvimento', 'A_alvo_reserva', 'estado_reserva', 'motivos']])
    display(AUD_RESERVA[AUD_RESERVA.bloco == ALVO])
    if (AUD_RESERVA.n_avaliaveis == 0).any(): AVISOS.append('Reserva: pelo menos um bloco/placa sem controles avaliaveis; nao interpretar como 0% de excedencia.')
    if CFG['reserva_ja_consultada']: print('Esta reserva ja havia sido consultada: resultado retrospectivo, nao novo teste intocado.')
else:
    print('Reserva nao aberta nesta execucao. Desenvolvimento, modelos e identidades permanecem disponiveis.')

## 19. Fechar o registro da investigação

O relatório abaixo é preenchido **somente pelos resultados calculados nesta execução**. Ele não contém nomes ou contagens esperadas. Os arquivos-fonte são novamente conferidos para detectar alterações durante a análise. As tabelas completas permitem examinar tanto candidatos quanto condições não priorizadas e não avaliáveis.

O pacote de leitura inclui relatório, tabelas e figuras. Matrizes, modelos e arquivos SHAP completos permanecem na pasta da execução, evitando duplicar grandes arquivos no ZIP. Para retomar, use a pasta original em `PASTA_RETOMAR`, com configuração e fontes inalteradas; os ajustes concluídos serão verificados por hash.

**Como concluir biologicamente:** uma assinatura é uma hipótese derivada dos perfis, não uma demonstração de disfunção. A identificação documental não estabelece mecanismo na linhagem. Um grande escore pode ser dominado por poucas medidas ou poucos objetos. Predizer bem a magnitude não garante reconhecer todos os candidatos, e a ausência de candidatos não autoriza ajustar cortes até aparecerem nomes desejados.

**Preparação para quatro locais:** as tabelas recebem a coluna `conjunto`; `manifesto_para_comparacao.json` identifica a configuração e as saídas desta execução. As matrizes por condição são exportadas com a ordem das medidas e das réplicas. Nenhuma comparação entre locais é calculada aqui. `id_condicao` é local: posteriormente, a correspondência deve usar identidade do composto e dose, com unidades verificadas.


In [ ]:
for path, h in ORIGINAIS.items(): exigir(sha256(path) == h, 'Arquivo-fonte modificado: ' + path)
resumo = [
    '# Analise de Cell Painting: ' + CFG['conjunto'],
    f'Canal-alvo: {ALVO}. Desenvolvimento: {", ".join(REPS)}.',
    f'Perfis de desenvolvimento incluidos: {len(meta)}; medidas iniciais: {len(NOMES)}.',
    f'Medidas selecionadas: {len(INDICES)}; no alvo: {len(NOMES_ALVO)}.',
    f'Condicoes examinadas: {len(condicoes)}; completas para os escores: {len(validos)}.',
    f'Candidatos descobertos a partir de todas as condicoes: {len(candidatos)}.',
    'Nenhuma lista anterior de candidatos foi utilizada. Identidade ausente nao exclui candidato.',
    '## Candidatos desta execucao'
]
for row in CANDIDATOS_IDENTIFICADOS.itertuples():
    resumo.append(f'- {row.composto} | {row.name or "nome nao resolvido"} | placa {row.placa} | dose {row.dose} (unidade conforme fonte).')
if not len(candidatos): resumo.append('Nenhuma condicao atendeu a regra; nenhum criterio foi relaxado automaticamente.')
if len(METRICAS):
    resumo += ['## Predicao condicional entre replicas', METRICAS.to_string(index=False),
               'Comparacao por composto omitido do treinamento, condicionada ao alvo e coorte previamente definidos.',
               'Nao e validacao externa, classificacao de mecanismo ou transferencia de modelo entre locais.']
else: resumo.append('Modelagem nao executada nesta configuracao.')
resumo += ['## Replica reservada']
if ABRIR_RESERVADA and CFG['reservada']:
    estados = AVALIACAO_RESERVA.estado_reserva.value_counts().to_dict()
    resumo += [str(estados), 'Lista e referencias da descoberta permaneceram fixas.',
        'Estado previo da reserva: ' + ('ja consultada; reanalise' if CFG['reserva_ja_consultada'] else 'declarada nao consultada antes desta campanha')]
else: resumo.append('Nao avaliada: abertura desativada ou nenhuma reserva declarada.')
n_baixa_contagem = int(meta.alerta_poucos_objetos.sum())
if n_baixa_contagem:
    AVISOS.append(f'{n_baixa_contagem} pocos de desenvolvimento com menos de {CFG["alerta_objetos_abaixo"]} objetos: apenas alerta, salvo filtro declarado.')
resumo += ['## Alertas', '\n'.join('- ' + a for a in AVISOS) or 'Nenhum alerta adicional; consulte as tabelas completas de qualidade, contagens e dominancia.',
    'Baixa contagem de objetos e dominancia sao diagnosticos, nao comprovacao de toxicidade ou falha de segmentacao.',
    'MAD invalido nao foi substituido; escores nao avaliaveis nao foram convertidos em negativos.',
    'SHAP explica as predições dos modelos, nao causalidade. Anotacoes foram usadas somente apos a modelagem.',
    'As analises suplementares historicas de ZCA e calibracao residual nao foram repetidas neste roteiro.']
RELATORIO = '\n\n'.join(resumo)
(SAIDA / 'relatorio.md').write_text(RELATORIO, encoding='utf-8')
(SAIDA / 'legendas.md').write_text('\n\n'.join('## ' + k + '\n' + v for k, v in LEGENDAS.items()), encoding='utf-8')
gravar_json(SAIDA / 'conclusao_execucao.json', {'codigo_versao': CFG['versao'], 'protocolo_sha256': HASH_PROTOCOLO,
    'descoberta_sha256': HASH_DESCOBERTA, 'fontes_inalteradas': True,
    'n_candidatos': len(candidatos), 'n_modelos': len(AUDITORIA_MODELOS),
    'reserva_aberta': bool(ABRIR_RESERVADA and CFG['reservada']), 'avisos': AVISOS})
# Saidas explicitas para comparacao futura; nao combinam locais nesta execucao.
np.savez_compressed(SAIDA / 'condicoes_desenvolvimento.npz',
    A=A_CUBO, Z_alvo=Z_ALVO, ids_condicao=condicoes.id_condicao.to_numpy(),
    blocos=np.asarray(BLOCOS, dtype=str), medidas_alvo=np.asarray(NOMES_ALVO, dtype=str),
    replicas=np.asarray(REPS, dtype=str), conjunto=np.asarray(CFG['conjunto']))
tabela('16_indice_condicoes', condicoes[['id_condicao'] + CHAVE + ['dose', 'avaliavel', 'candidato']])
comparacao = {'schema': 'cellpainting-local-1.0', 'conjunto': CFG['conjunto'],
    'versao': CFG['versao'], 'nucleo_sha256': CFG['nucleo_sha256'],
    'protocolo_sha256': HASH_PROTOCOLO, 'descoberta_sha256': HASH_DESCOBERTA,
    'origem_perfis': URL_PASTA_DADOS, 'treinamento': 'exclusivamente dentro deste local',
    'replicas_desenvolvimento': REPS, 'replica_reservada': CFG['reservada'],
    'reserva_aberta': bool(ABRIR_RESERVADA and CFG['reservada']),
    'coorte_condicoes': len(condicoes), 'coorte_modelagem': len(META_COMPLETO),
    'alvo': ALVO, 'n_medidas_alvo': len(NOMES_ALVO),
    'definicao_alvo': 'log(1+RMS/P95); medidas e P95 definidos neste local em desenvolvimento',
    'unidade_dose': 'preservada da fonte; conferir antes de comparar locais',
    'comparacao_entre_locais_executada': False,
    'arquivos_chave': ['tabelas/06_todas_condicoes.csv', 'tabelas/04_catalogo_selecao.csv',
      'tabelas/05_referencias_congeladas.csv', 'tabelas/02_contagens_pocos.csv',
      'tabelas/05_dominancia_por_poco.csv', 'condicoes_desenvolvimento.npz',
      'tabelas/12_predicoes_externas.csv', 'tabelas/13_SHAP_global.csv',
      'tabelas/13_SHAP_grupos_com_sinal.csv', 'tabelas/14_identidades_toda_coorte.csv',
      'tabelas/15_avaliacao_candidatos_reserva.csv'],
    'nota': 'Ausencia de resultado ou nao avaliavel nao significa resultado negativo. Alvos locais nao sao unidades biologicas comuns.'}
comparacao['arquivos_existentes'] = {p: sha256(SAIDA / p) for p in comparacao['arquivos_chave'] if (SAIDA / p).is_file()}
gravar_json(SAIDA / 'manifesto_para_comparacao.json', comparacao)

PACOTE = SAIDA.parent / (SAIDA.name + '_leitura.zip')
with zipfile.ZipFile(PACOTE, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in SAIDA.rglob('*'):
        if p.is_file() and (p.parent.name in ['tabelas', 'figuras'] or p.suffix in ['.md', '.json'] and p.parent == SAIDA):
            z.write(p, p.relative_to(SAIDA))
display(Markdown(RELATORIO))
print('RESULTADOS:', SAIDA)
print('PACOTE DE LEITURA:', PACOTE)

## Referências e origem das decisões

O desenho principal segue os procedimentos documentados no TCC de Mario Costa Cruz: desenvolvimento nas réplicas declaradas, normalização por DMSO, seleção de características, escore RMS/P95 e avaliação separada dos candidatos. O número de candidatos é um resultado, não uma entrada. Este caderno é uma implementação didática nova; não substitui retroativamente as execuções arquivadas. Pequenas diferenças de versões ou de partições devem ser registradas, não ocultadas.

As tabelas documentais de IMTM_HepG2 foram copiadas do arquivo dos autores de **EU-OS bioactives**; os nomes de seus arquivos identificam snapshots, não farmacologia atualizada. O mapa de outro experimento precisa ser fornecido pelo pesquisador.

- Wolff et al. Repositório e dados de EU-OS bioactives: https://github.com/schmiedc/EU-OS_bioactives
- Caicedo et al. (2017). Data-analysis strategies for image-based cell profiling. https://doi.org/10.1038/nmeth.4397
- XGBoost, API usada para regressão e parada antecipada: https://xgboost.readthedocs.io/en/release_3.1.0/python/python_api.html
- scikit-learn, separação por grupos: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html
- scikit-learn, vazamento de informação e pré-processamento: https://scikit-learn.org/stable/common_pitfalls.html
- SHAP, explicações TreeExplainer: https://shap.readthedocs.io/en/latest/generated/shap.TreeExplainer.html
- Significado das famílias de medidas CellProfiler: https://cellprofiler-manual.s3.amazonaws.com/CellProfiler-4.2.4/modules/measurement.html

**Glossário curto:** perfil = conjunto de medidas de um poço; condição = placa + composto + dose; bloco = conjunto de medidas; MAD = dispersão robusta dos controles; RMS = magnitude quadrática; P95 = referência de cauda; candidato = condição que atendeu à regra; fold = partição de treinamento/avaliação; OOF = previsão de um exemplo fora do treinamento do modelo; SHAP = atribuições para explicar uma previsão, não um mecanismo biológico.

**Organização das saídas:** `06_todas_condicoes.csv` documenta as decisões, `07_candidatos_e_bootstrap.csv` contém a lista calculada nesta execução, e as tabelas com prefixo `14_` associam identidades sem filtrar os resultados. Os arquivos `15_` são produzidos somente quando a avaliação reservada é liberada. Consulte os nomes efetivamente impressos na pasta da execução.

**Adaptação nesta edição:** leitura de metadados específica dos novos locais; isolamento de caminhos e retomada; identificação do local nas saídas e manifesto para a comparação futura. O código analítico de descoberta, modelos, hiperparâmetros e SHAP foi preservado em relação ao roteiro didático 1.0. As execuções antigas de FMP/IMTM não são sobrescritas.


**Padronização de 03/10/2026:** cálculos numéricos idênticos aos das seções 5–18 de MEDINA/USC (descontada a chave de abertura de R4). As mudanças de código se limitam à adaptação dos metadados, à proveniência dos arquivos e à configuração local. A padronização não uniformiza automaticamente aquisição, cultura celular, coortes ou espaços fenotípicos. Seleção e P95 continuam estimados separadamente.
